In [ ]:
# TensorFlowを読み込む
import tensorflow as tf

# TensorFlowのバージョンを確認する
print("TensorFlowのバージョン：", tf.__version__)

# TensorFlowが認識しているGPUを確認する
gpus = tf.config.list_physical_devices("GPU")

print("認識されたGPU：", gpus)
print("GPUの数：", len(gpus))

##### 今回はGPUなしで試す。

In [ ]:
# ファイルやフォルダの場所を扱う
from pathlib import Path

# Excelや表形式データを扱う
import pandas as pd

# 表を見やすく表示する
from IPython.display import display


### 統計用データを読み込む

In [ ]:
# 統計用Excelが保存されているフォルダを指定する
excel_dir = Path(
    "/home/jovyan/work/ssd/dermatology"
)

# サブフォルダを含めて「統計用データ.xlsx」を検索する
excel_files = list(
    excel_dir.rglob("統計用データ.xlsx")
)

# 見つかったファイルの保存場所を表示する
for file in excel_files:
    print(file)

# ファイルが1件だけ見つかったことを確認する
if len(excel_files) != 1:
    raise ValueError(
        f"{len(excel_files)}件見つかりました。"
        "読み込むファイルの場所を確認してください。"
    )

# 「統計用データ」シートを読み込む
df_stats = pd.read_excel(
    excel_files[0],
    sheet_name="統計用データ",
    header=0,
    dtype=object,
    engine="openpyxl"
)

# 読み込んだデータの大きさと列名を確認する
print("行数・列数：", df_stats.shape)
print("列名：", df_stats.columns.tolist())

In [ ]:
df_stats

### EASI画像の一覧を作る

In [ ]:
# EASI画像が保存されている一番上のフォルダを指定する
data_root = Path(
    "/home/jovyan/work/ssd/easi_data/dermatology_ml"
)

# データフォルダが存在することを確認する
if not data_root.exists():
    raise FileNotFoundError(
        f"EASI画像フォルダが見つかりません：{data_root}"
    )

# EASIで評価する4つの身体部位
regions = ["head", "upper", "trunk", "lower"]

# 画像ファイルとして扱う拡張子
image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
    ".heic",
    ".heif"
}

# head・upper・trunk・lowerフォルダを検索する
region_dirs = [
    folder
    for folder in data_root.rglob("*")
    if folder.is_dir()
    and folder.name.lower() in regions
    and not any(
        part.startswith(".") or part == "__MACOSX"
        for part in folder.relative_to(data_root).parts
    )
]

# 画像1枚につき1行の一覧を作る
records = []

for folder in region_dirs:
    for file in folder.rglob("*"):
        if (
            file.is_file()
            and file.suffix.lower() in image_extensions
            and not any(
                part.startswith(".") or part == "__MACOSX"
                for part in file.relative_to(folder).parts
            )
        ):
            records.append({
                "patient_id": folder.parent.name,
                "region": folder.name.lower(),
                "image_path": str(file)
            })

# 画像一覧をDataFrameにする
image_df = pd.DataFrame(
    records,
    columns=[
        "patient_id",
        "region",
        "image_path"
    ]
)

# 画像一覧の内容を確認する
print("部位フォルダ数：", len(region_dirs))
print(
    "患者フォルダ名の種類数：",
    image_df["patient_id"].nunique()
)
print("画像ファイル数：", len(image_df))

display(image_df.head())

### IDの形式をそろえる

In [ ]:
# 元データを変更しないようコピーを作る
stats = df_stats.copy()
images = image_df.copy()

# ExcelのIDと画像フォルダ名を文字列型にそろえ、
# 前後の不要な空白を取り除く
stats["ID"] = (
    stats["ID"]
    .astype("string")
    .str.strip()
)

images["patient_id"] = (
    images["patient_id"]
    .astype("string")
    .str.strip()
)

# IDが空文字の場合は欠損値として扱う
stats["ID"] = stats["ID"].replace("", pd.NA)
images["patient_id"] = images["patient_id"].replace("", pd.NA)

# 欠損値を除き、重複のないIDの集合を作る
excel_ids = set(
    stats["ID"].dropna()
)

image_ids = set(
    images["patient_id"].dropna()
)

# Excelと画像の両方に存在するID
matched_ids = sorted(
    excel_ids & image_ids
)

# Excel側だけに存在するID
only_excel_ids = sorted(
    excel_ids - image_ids
)

# 画像側だけに存在する患者フォルダ名
only_image_ids = sorted(
    image_ids - excel_ids
)

# IDの種類数と一致数を表示する
print("Excel側のID種類数：", len(excel_ids))
print("画像側の患者フォルダ名種類数：", len(image_ids))
print("Excelと画像の両方に存在するID数：", len(matched_ids))

# 不一致の件数を表示する
print("Excel側だけに存在するID数：", len(only_excel_ids))
print(
    "画像側だけに存在する患者フォルダ名数：",
    len(only_image_ids)
)

# 不一致のIDを具体的に表示する
print("\n【Excel側だけに存在するID】")
display(only_excel_ids)

print("\n【画像側だけに存在する患者フォルダ名】")
display(only_image_ids)

In [ ]:
# Excel側でIDが欠損している行数を確認する
excel_missing_id_rows = stats[
    stats["ID"].isna()
]

print(
    "\nExcel側でIDが空欄の行数：",
    len(excel_missing_id_rows)
)

# IDが空欄の行について、紅斑を含む列だけを確認する
erythema_columns = [
    column
    for column in stats.columns
    if "紅斑" in str(column)
]

if not excel_missing_id_rows.empty:
    print("\n【IDが空欄の行の紅斑データ】")

    display(
        excel_missing_id_rows[
            ["ID"] + erythema_columns
        ]
    )

In [ ]:
import pandas as pd

# Excelの列名と画像フォルダの部位名を対応させる
region_map = {
    "頭部*紅斑": "head",
    "上肢*紅斑2": "upper",
    "体幹*紅斑2": "trunk",
    "下肢*紅斑2": "lower",
}

# 必要な列が存在するか確認
required_cols = ["ID"] + list(region_map.keys())

missing_cols = [
    col for col in required_cols
    if col not in stats.columns
]

if missing_cols:
    raise ValueError(f"存在しない列: {missing_cols}")

# Excelの横長データを縦長データに変換する
erythema_labels = (
    stats[required_cols]
    .dropna(subset=["ID"])
    .rename(columns={"ID": "patient_id"})
    .melt(
        id_vars="patient_id",
        value_vars=list(region_map.keys()),
        var_name="region",
        value_name="erythema_score"
    )
)

# IDの形式を統一する
erythema_labels["patient_id"] = (
    erythema_labels["patient_id"]
    .astype("string")
    .str.strip()
)

# 身体部位を英語表記に変換する
erythema_labels["region"] = (
    erythema_labels["region"].map(region_map)
)

# 紅斑スコアを数値に変換する
erythema_labels["erythema_score"] = pd.to_numeric(
    erythema_labels["erythema_score"],
    errors="coerce"
)

# 患者IDと身体部位の重複を確認する
duplicate_count = erythema_labels.duplicated(
    subset=["patient_id", "region"]
).sum()

print("重複数:", duplicate_count)
print("患者数:", erythema_labels["patient_id"].nunique())
print("ラベル行数:", len(erythema_labels))

display(erythema_labels.head(10))

In [ ]:
# IDが空欄の行を除外する
erythema_labels = (
    erythema_labels
    .dropna(subset=["patient_id"])
    .reset_index(drop=True)
)

# 同じ患者・同じ部位の重複数を確認する
duplicate_count = erythema_labels.duplicated(
    subset=["patient_id", "region"]
).sum()

print("ID空欄行を除外した後の重複数：", duplicate_count)

### 画像に紅斑のスコアをつける

In [ ]:
# 元の画像一覧を変更しないようコピーを作る
images_for_model = image_df.copy()

# 画像側の患者フォルダ名を文字列型にそろえ、
# 前後の不要な空白を取り除く
images_for_model["patient_id"] = (
    images_for_model["patient_id"]
    .astype("string")
    .str.strip()
)

# 画像一覧と部位別の紅斑スコアを、
# patient_idとregionを使って結合する
erythema_df = images_for_model.merge(
    erythema_labels,
    on=["patient_id", "region"],
    how="left",
    validate="many_to_one"
)

# モデル作成に必要な列だけを指定した順番に並べる
erythema_df = erythema_df[
    [
        "patient_id",
        "region",
        "image_path",
        "erythema_score"
    ]
]

# 結合結果の先頭10行を確認する
display(erythema_df.head(10))

In [ ]:
# 紅斑スコアが付かなかった画像を取り出す
missing_erythema = erythema_df[
    erythema_df["erythema_score"].isna()
].copy()

# 欠損件数を確認する
print(
    "紅斑スコアが付かなかった画像数：",
    len(missing_erythema)
)

print(
    "紅斑スコアが付かなかった患者フォルダ数：",
    missing_erythema["patient_id"].nunique()
)

# 欠損している画像の先頭20行を確認する
display(missing_erythema.head(20))

In [ ]:
# Excel側に存在するIDの集合を作る
label_patient_ids = set(
    erythema_labels["patient_id"].dropna()
)

# Excel側にID自体が存在しない画像
missing_id_images = missing_erythema[
    ~missing_erythema["patient_id"].isin(label_patient_ids)
].copy()

# Excel側にIDはあるが、部位別紅斑スコアがない画像
missing_score_images = missing_erythema[
    missing_erythema["patient_id"].isin(label_patient_ids)
].copy()

print(
    "Excel側にIDがない画像数：",
    len(missing_id_images)
)

print(
    "Excel側にIDはあるが紅斑スコアがない画像数：",
    len(missing_score_images)
)

print("\n【Excel側にIDがない画像】")
display(
    missing_id_images[
        ["patient_id", "region", "image_path"]
    ].head(20)
)

print("\n【IDはあるが紅斑スコアがない画像】")
display(
    missing_score_images[
        ["patient_id", "region", "image_path"]
    ].head(20)
)

In [ ]:
# 紅斑スコアを数値型に変換する
# 数値に変換できない値や空欄はNaNにする
erythema_df["erythema_score"] = pd.to_numeric(
    erythema_df["erythema_score"],
    errors="coerce"
)

# 紅斑スコアが0～3点の画像だけを残す
erythema_df = erythema_df[
    erythema_df["erythema_score"].isin(
        [0, 1, 2, 3]
    )
].copy()

# 紅斑スコアを整数型へ変換する
erythema_df["erythema_score"] = (
    erythema_df["erythema_score"]
    .astype("int32")
)

# 行番号を0から振り直す
erythema_df = erythema_df.reset_index(
    drop=True
)

print(
    "紅斑モデルに使用できる画像数：",
    len(erythema_df)
)

print(
    "紅斑モデルに使用できる患者フォルダ数：",
    erythema_df["patient_id"].nunique()
)

display(erythema_df.head(10))

In [ ]:
# 紅斑スコアごとの画像数を確認する
score_distribution = (
    erythema_df["erythema_score"]
    .value_counts()
    .sort_index()
    .rename_axis("紅斑スコア")
    .reset_index(name="画像数")
)

print("【紅斑スコア別の画像数】")
display(score_distribution)

In [ ]:
# 部位別・紅斑スコア別の画像数を確認する
region_score_distribution = pd.crosstab(
    erythema_df["region"],
    erythema_df["erythema_score"]
).reindex(
    index=["head", "upper", "trunk", "lower"],
    columns=[0, 1, 2, 3],
    fill_value=0
)

print("【部位別・紅斑スコア別の画像数】")
display(region_score_distribution)

### 画像データを学習用、検証用、テスト用に分割する

#### 画像枚数の差について

In [ ]:
# 患者ごとの画像数を確認する
images_per_patient = (
    erythema_df.groupby("patient_id")
    .size()
    .sort_values(ascending=False)
)

print("患者数：", len(images_per_patient))
print("患者あたり最小画像数：", images_per_patient.min())
print("患者あたり中央値：", images_per_patient.median())
print("患者あたり平均画像数：", images_per_patient.mean())
print("患者あたり最大画像数：", images_per_patient.max())

display(
    images_per_patient
    .rename("画像数")
    .head(20)
)

In [ ]:
# 患者・部位ごとの画像数を確認する
images_per_patient_region = (
    erythema_df.groupby(
        ["patient_id", "region"]
    )
    .size()
    .rename("画像数")
    .reset_index()
)

display(
    images_per_patient_region
    .sort_values("画像数", ascending=False)
    .head(30)
)

#### 患者単位の候補分割を複数作る

In [ ]:
# 患者・部位ごとの画像数を集計する
images_per_patient_region = (
    erythema_df.groupby(
        ["patient_id", "region"]
    )
    .size()
    .rename("画像数")
    .reset_index()
)

# 画像数の分布を確認する
print(
    images_per_patient_region["画像数"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

In [ ]:
# 同じ患者の画像が複数のデータ群に混ざらないよう、
# patient_id単位で学習用・検証用・テスト用に分割する

from sklearn.model_selection import GroupShuffleSplit


# 患者単位で学習用70%と一時データ30%に分ける
first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_index, temp_index = next(
    first_split.split(
        erythema_df,
        groups=erythema_df["patient_id"]
    )
)

train_df = (
    erythema_df
    .iloc[train_index]
    .copy()
    .reset_index(drop=True)
)

temp_df = (
    erythema_df
    .iloc[temp_index]
    .copy()
    .reset_index(drop=True)
)


# 一時データ30%を、検証用15%とテスト用15%に分ける
second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_index, test_index = next(
    second_split.split(
        temp_df,
        groups=temp_df["patient_id"]
    )
)

val_df = (
    temp_df
    .iloc[val_index]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    temp_df
    .iloc[test_index]
    .copy()
    .reset_index(drop=True)
)

In [ ]:
# 各データ群の患者IDを集合にする
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])

# 同じ患者が複数のデータ群に混ざっていないことを確認する
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

print("患者単位で重複なく分割できています。")

print(
    "学習用：",
    len(train_df), "画像 /",
    len(train_patients), "患者"
)

print(
    "検証用：",
    len(val_df), "画像 /",
    len(val_patients), "患者"
)

print(
    "テスト用：",
    len(test_df), "画像 /",
    len(test_patients), "患者"
)

In [ ]:
# 学習用データについて、
# 同じ患者・同じ部位から使用する画像を最大10枚に制限する
MAX_IMAGES_PER_PATIENT_REGION = 10

train_model_df = (
    train_df
    .groupby(
        ["patient_id", "region"],
        group_keys=False
    )
    .apply(
        lambda group: group.sample(
            n=min(
                len(group),
                MAX_IMAGES_PER_PATIENT_REGION
            ),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("枚数制限前の学習画像数：", len(train_df))
print("枚数制限後の学習画像数：", len(train_model_df))

print(
    "学習用患者数：",
    train_model_df["patient_id"].nunique()
)

In [ ]:
# 患者・部位あたりの画像数が10枚以下になったことを確認する
train_group_counts = (
    train_model_df
    .groupby(["patient_id", "region"])
    .size()
)

print(
    "制限後の患者・部位あたり最大画像数：",
    train_group_counts.max()
)

assert (
    train_group_counts.max()
    <= MAX_IMAGES_PER_PATIENT_REGION
)

print("患者・部位あたり最大10枚に制限できています。")

In [ ]:
# 枚数制限前後の紅斑スコア別画像数を比較する
score_comparison = pd.DataFrame({
    "制限前": (
        train_df["erythema_score"]
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "制限後": (
        train_model_df["erythema_score"]
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

score_comparison.index.name = "紅斑スコア"

display(score_comparison)

In [ ]:
# 学習・検証・テストの紅斑スコア分布を確認する
split_distribution = pd.DataFrame({
    "学習用": (
        train_model_df["erythema_score"]
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "検証用": (
        val_df["erythema_score"]
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "テスト用": (
        test_df["erythema_score"]
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

split_distribution.index.name = "紅斑スコア"

display(split_distribution)

### 選ばれた画像を確認する

In [ ]:
# 確認する患者と部位を指定する
check_patient = "197"
check_region = "lower"

# 制限前の画像
before_selection = train_df[
    (train_df["patient_id"] == check_patient)
    & (train_df["region"] == check_region)
][
    ["patient_id", "region", "image_path", "erythema_score"]
].sort_values("image_path")

# 最大10枚に制限した後の画像
after_selection = train_model_df[
    (train_model_df["patient_id"] == check_patient)
    & (train_model_df["region"] == check_region)
][
    ["patient_id", "region", "image_path", "erythema_score"]
].sort_values("image_path")

print("制限前の画像数：", len(before_selection))
print("選ばれた画像数：", len(after_selection))

print("\n【選ばれた画像】")
display(after_selection)

### 学習データセットの作成

#### 画像形式の件数の確認

In [ ]:
# 学習・検証・テストに含まれる画像形式を確認する
def count_extensions(dataframe):
    return (
        dataframe["image_path"]
        .map(lambda path: Path(path).suffix.lower())
        .value_counts()
        .sort_index()
    )


extension_summary = pd.DataFrame({
    "学習用": count_extensions(train_model_df),
    "検証用": count_extensions(val_df),
    "テスト用": count_extensions(test_df)
}).fillna(0).astype(int)

extension_summary.index.name = "拡張子"

display(extension_summary)

In [ ]:
# TensorFlowで使用する画像データセットを作成する

import tensorflow as tf

# MobileNetV2へ入力する画像サイズ
IMG_SIZE = 224

# 1回の処理でモデルへ渡す画像数
BATCH_SIZE = 16

# データの読み込み処理を自動的に並列化する
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
def load_image(path, label):
    """JPEG画像と紅斑スコアをモデル入力用に整える"""

    # 画像ファイルを読み込む
    image_data = tf.io.read_file(path)

    # JPEG画像をRGB形式で展開する
    image = tf.image.decode_jpeg(
        image_data,
        channels=3
    )

    # 画像サイズを224×224へ統一する
    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    # 画素値を小数型へ変換する
    image = tf.cast(
        image,
        tf.float32
    )

    # 紅斑スコアを整数型へ変換する
    label = tf.cast(
        label,
        tf.int32
    )

    return image, label

In [ ]:
# 学習用の画像パスと紅斑スコアを取り出す
train_paths = (
    train_model_df["image_path"]
    .astype(str)
    .to_numpy()
)

train_labels = (
    train_model_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)

# 検証用の画像パスと紅斑スコアを取り出す
val_paths = (
    val_df["image_path"]
    .astype(str)
    .to_numpy()
)

val_labels = (
    val_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)

# テスト用の画像パスと紅斑スコアを取り出す
test_paths = (
    test_df["image_path"]
    .astype(str)
    .to_numpy()
)

test_labels = (
    test_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)

In [ ]:
# 画像パスと紅斑スコアをTensorFlowデータセットへ変換する
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_paths, test_labels)
)

In [ ]:
# 学習用データセットを整える
train_ds = (
    train_ds

    # 各エポックで画像の処理順を入れ替える
    .shuffle(
        buffer_size=len(train_model_df),
        seed=42,
        reshuffle_each_iteration=True
    )

    # JPEG画像を読み込み、サイズと型をそろえる
    .map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    # 画像を16枚ずつまとめる
    .batch(BATCH_SIZE)

    # 次のバッチを事前に準備して読み込みを高速化する
    .prefetch(AUTOTUNE)
)

In [ ]:
# 検証用データセットを整える
val_ds = (
    val_ds
    .map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# テスト用データセットを整える
test_ds = (
    test_ds
    .map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
# 学習用データセットから1バッチを取り出して確認する
for image_batch, label_batch in train_ds.take(1):
    print(
        "画像バッチの形：",
        image_batch.shape
    )

    print(
        "ラベルバッチの形：",
        label_batch.shape
    )

    print(
        "バッチ内の紅斑スコア：",
        label_batch.numpy()
    )

    print(
        "画素値の最小値：",
        float(tf.reduce_min(image_batch))
    )

    print(
        "画素値の最大値：",
        float(tf.reduce_max(image_batch))
    )

### 実際の画像を確認

In [ ]:
# 学習用データから画像と紅斑スコアを表示し、
# 画像と正解ラベルの対応を確認する

! pip install japanize_matplotlib #日本語表記をできるようにする。
import matplotlib.pyplot as plt
import japanize_matplotlib

# 紅斑スコアの表示名
class_names = {
    0: "紅斑 0：なし",
    1: "紅斑 1：軽度",
    2: "紅斑 2：中等度",
    3: "紅斑 3：高度"
}

# 3行×3列の画像表示領域を作る
plt.figure(figsize=(12, 12))

# 学習用データセットから最初の1バッチを取り出す
for images, labels in train_ds.take(1):

    # バッチ内の先頭9枚を表示する
    for i in range(min(9, len(images))):

        # 3行×3列のうち、現在の表示位置を指定する
        ax = plt.subplot(3, 3, i + 1)

        # 画像を表示する
        plt.imshow(
            images[i]
            .numpy()
            .clip(0, 255)
            .astype("uint8")
        )

        # 正解の紅斑スコアをタイトルに表示する
        label_value = int(labels[i].numpy())

        plt.title(
            class_names[label_value]
        )

        # 画像周囲の目盛りを非表示にする
        plt.axis("off")

# 画像同士の重なりを調整する
plt.tight_layout()

# 画像を表示する
plt.show()

### MobileNetV2モデルを作成

In [ ]:
# 学習用画像に加えるデータ拡張を定義する
data_augmentation = tf.keras.Sequential(
    [
        # 画像をランダムに左右反転する
        tf.keras.layers.RandomFlip(
            mode="horizontal"
        ),

        # 画像を約±10度の範囲でランダムに回転する
        tf.keras.layers.RandomRotation(
            factor=0.03
        ),

        # 画像を上下・左右へ最大5%ランダムに移動する
        tf.keras.layers.RandomTranslation(
            height_factor=0.05,
            width_factor=0.05
        ),

        # 画像を最大10%ランダムに拡大・縮小する
        tf.keras.layers.RandomZoom(
            height_factor=0.10,
            width_factor=0.10
        )
    ],
    name="data_augmentation"
)

In [ ]:
# ImageNetで学習済みのMobileNetV2を読み込む
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    ),
    include_top=False,
    weights="imagenet"
)

# 最初はMobileNetV2本体の重みを固定する
base_model.trainable = False


In [ ]:
# モデルへの画像入力を定義する
inputs = tf.keras.Input(
    shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    ),
    name="input_image"
)

# 学習時だけデータ拡張を適用する
x = data_augmentation(
    inputs
)

# MobileNetV2用に画素値を-1～1へ変換する
x = tf.keras.applications.mobilenet_v2.preprocess_input(
    x
)

# MobileNetV2で画像特徴を抽出する
# BatchNormalizationの更新を防ぐためtraining=Falseを指定する
x = base_model(
    x,
    training=False
)

# 画像の縦横方向の特徴を平均し、1次元へまとめる
x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

# 過学習を抑えるため、学習時に一部の特徴をランダムに無効化する
x = tf.keras.layers.Dropout(
    rate=0.30,
    name="dropout"
)(x)

# 紅斑スコア0・1・2・3の確率を出力する
outputs = tf.keras.layers.Dense(
    units=4,
    activation="softmax",
    name="erythema_output"
)(x)

# 入力から出力までを1つのモデルにまとめる
model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="erythema_mobilenetv2"
)

In [ ]:
# 4クラス分類モデルの学習方法を設定する
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

In [ ]:
# 作成したモデルの構造を表示する
model.summary()


### 学習

In [ ]:
# 学習済みモデルを保存するフォルダを指定する
model_dir = Path(
    "/home/jovyan/work/ssd/easi_data/models"
)

# 保存フォルダがなければ作成する
model_dir.mkdir(
    parents=True,
    exist_ok=True
)

# 学習済みの重みを保存するフォルダを指定する
model_dir = Path(
    "/home/jovyan/work/ssd/easi_data/models"
)

# 保存フォルダがなければ作成する
model_dir.mkdir(
    parents=True,
    exist_ok=True
)

# 検証損失が最も小さかった時点の重みを保存する場所
weights_path = (
    model_dir
    / "erythema_mobilenetv2_stage1.weights.h5"
)

print("重みの保存場所：", weights_path)

In [ ]:
# 学習を安全に進めるための処理を設定する
callbacks = [
    # 検証損失が5エポック改善しなければ学習を終了し、
    # 最も検証損失が小さかった時点の重みに戻す
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    # 検証損失が最も小さかった時点の重みだけを保存する
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(weights_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    # 検証損失が2エポック改善しなければ学習率を半分にする
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
# 学習に使用するデータ数を確認する
print("学習用画像数：", len(train_model_df))
print("検証用画像数：", len(val_df))
print("テスト用画像数：", len(test_df))

print(
    "学習用バッチ数：",
    tf.data.experimental.cardinality(train_ds).numpy()
)

print(
    "検証用バッチ数：",
    tf.data.experimental.cardinality(val_ds).numpy()
)

In [ ]:
# MobileNetV2本体を固定した状態で、
# 追加した紅斑分類部分を学習する
INITIAL_EPOCHS = 20

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks,
    verbose=1
)

### 評価

In [ ]:
# 第1段階の学習履歴を表示する

train_loss = history_stage1.history["loss"]
val_loss = history_stage1.history["val_loss"]

train_accuracy = history_stage1.history["accuracy"]
val_accuracy = history_stage1.history["val_accuracy"]

epochs_ran = range(
    1,
    len(train_loss) + 1
)

plt.figure(figsize=(14, 5))

# 損失の推移
plt.subplot(1, 2, 1)

plt.plot(
    epochs_ran,
    train_loss,
    marker="o",
    label="学習用"
)

plt.plot(
    epochs_ran,
    val_loss,
    marker="o",
    label="検証用"
)

plt.xlabel("エポック")
plt.ylabel("損失")
plt.title("第1段階：損失の推移")
plt.legend()

# 正解率の推移
plt.subplot(1, 2, 2)

plt.plot(
    epochs_ran,
    train_accuracy,
    marker="o",
    label="学習用"
)

plt.plot(
    epochs_ran,
    val_accuracy,
    marker="o",
    label="検証用"
)

plt.xlabel("エポック")
plt.ylabel("正解率")
plt.title("第1段階：正解率の推移")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 最良時点の重みが保存されているか確認する
print(
    "重みファイルの存在：",
    weights_path.is_file()
)

print(
    "重みの保存場所：",
    weights_path
)

In [ ]:
import numpy as np

# 検証用画像について、紅斑スコア0～3の予測確率を取得する
val_probabilities = model.predict(
    val_ds,
    verbose=1
)

# 最も確率が高いスコアを予測結果にする
val_predictions = np.argmax(
    val_probabilities,
    axis=1
)

# 正解ラベルをNumPy配列へ変換する
val_true = (
    val_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)

print("検証画像数：", len(val_true))
print("予測数：", len(val_predictions))

print(
    "予測スコアの種類：",
    np.unique(
        val_predictions,
        return_counts=True
    )
)

### 混合行列

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# 正解スコアと予測スコアの組み合わせを集計する
val_confusion_matrix = confusion_matrix(
    val_true,
    val_predictions,
    labels=[0, 1, 2, 3]
)

val_confusion_df = pd.DataFrame(
    val_confusion_matrix,
    index=[
        "正解0",
        "正解1",
        "正解2",
        "正解3"
    ],
    columns=[
        "予測0",
        "予測1",
        "予測2",
        "予測3"
    ]
)

display(val_confusion_df)


In [ ]:
from sklearn.metrics import classification_report

# 紅斑スコア別の適合率・再現率・F1スコアを表示する
print(
    classification_report(
        val_true,
        val_predictions,
        labels=[0, 1, 2, 3],
        target_names=[
            "紅斑0",
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    cohen_kappa_score
)

# 予測と正解の平均的な点数差
val_mae = mean_absolute_error(
    val_true,
    val_predictions
)

# 正解との差が1点以内だった画像の割合
val_within_one = np.mean(
    np.abs(
        val_true - val_predictions
    ) <= 1
)

# 順序を考慮した一致度
val_qwk = cohen_kappa_score(
    val_true,
    val_predictions,
    weights="quadratic"
)

print("検証MAE：", round(val_mae, 3))
print(
    "検証1点以内正解率：",
    round(val_within_one, 3)
)
print(
    "検証Quadratic Weighted Kappa：",
    round(val_qwk, 3)
)

### 学習曲線

In [ ]:
# 第1段階の学習結果をグラフで確認する

# 学習履歴を取り出す
train_loss = history_stage1.history["loss"]
val_loss = history_stage1.history["val_loss"]

train_accuracy = history_stage1.history["accuracy"]
val_accuracy = history_stage1.history["val_accuracy"]

# 実際に学習したエポック数
epochs_ran = range(
    1,
    len(train_loss) + 1
)

# 損失と正解率を横並びで表示する
plt.figure(figsize=(14, 5))


# 損失の推移
plt.subplot(1, 2, 1)

plt.plot(
    epochs_ran,
    train_loss,
    marker="o",
    label="学習用"
)

plt.plot(
    epochs_ran,
    val_loss,
    marker="o",
    label="検証用"
)

plt.xlabel("エポック")
plt.ylabel("損失")
plt.title("第1段階：損失の推移")
plt.legend()
plt.grid(alpha=0.3)


# 正解率の推移
plt.subplot(1, 2, 2)

plt.plot(
    epochs_ran,
    train_accuracy,
    marker="o",
    label="学習用"
)

plt.plot(
    epochs_ran,
    val_accuracy,
    marker="o",
    label="検証用"
)

plt.xlabel("エポック")
plt.ylabel("正解率")
plt.title("第1段階：正解率の推移")
plt.legend()
plt.grid(alpha=0.3)

# グラフ同士の重なりを調整する
plt.tight_layout()
plt.show()

In [ ]:
# 最良時点の重みが保存されたことを確認する
print(
    "重みファイルの存在：",
    weights_path.is_file()
)

print(
    "重みの保存場所：",
    weights_path
)

In [ ]:
# 数値計算用ライブラリを読み込む
import numpy as np

# 検証用画像について、
# 紅斑スコア0～3それぞれの予測確率を取得する
val_probabilities = model.predict(
    val_ds,
    verbose=1
)

# 最も確率が高い紅斑スコアを予測結果とする
val_predictions = np.argmax(
    val_probabilities,
    axis=1
)

# 検証用データの正解スコアを取り出す
val_true = (
    val_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)

# 正解数と予測数が一致することを確認する
print("検証画像数：", len(val_true))
print("予測結果数：", len(val_predictions))

assert len(val_true) == len(val_predictions)

print("検証画像数と予測結果数が一致しています。")

In [ ]:
# 正解スコアと予測スコアの件数を比較する
prediction_distribution = pd.DataFrame({
    "正解画像数": (
        pd.Series(val_true)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "予測画像数": (
        pd.Series(val_predictions)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

prediction_distribution.index.name = "紅斑スコア"

display(prediction_distribution)

In [ ]:
# 混同行列を作成する
from sklearn.metrics import confusion_matrix

val_confusion_matrix = confusion_matrix(
    val_true,
    val_predictions,
    labels=[0, 1, 2, 3]
)

# 見やすい表へ変換する
val_confusion_df = pd.DataFrame(
    val_confusion_matrix,
    index=[
        "正解0",
        "正解1",
        "正解2",
        "正解3"
    ],
    columns=[
        "予測0",
        "予測1",
        "予測2",
        "予測3"
    ]
)

display(val_confusion_df)

In [ ]:
# 紅斑スコア別の評価結果を表示する
from sklearn.metrics import classification_report

print(
    classification_report(
        val_true,
        val_predictions,
        labels=[0, 1, 2, 3],
        target_names=[
            "紅斑0",
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
# 紅斑スコアの順序を考慮した評価指標を計算する
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    cohen_kappa_score
)

# 完全に一致した割合
val_accuracy_score = accuracy_score(
    val_true,
    val_predictions
)

# 正解と予測が平均で何点離れているか
val_mae = mean_absolute_error(
    val_true,
    val_predictions
)

# 正解との差が1点以内だった割合
val_within_one = np.mean(
    np.abs(
        val_true - val_predictions
    ) <= 1
)

# スコア間の距離を考慮した一致度
val_qwk = cohen_kappa_score(
    val_true,
    val_predictions,
    weights="quadratic"
)

print(
    "検証Accuracy：",
    round(val_accuracy_score, 3)
)

print(
    "検証MAE：",
    round(val_mae, 3)
)

print(
    "検証1点以内正解率：",
    round(val_within_one, 3)
)

print(
    "検証Quadratic Weighted Kappa：",
    round(val_qwk, 3)
)

In [ ]:
# 検証用データへ予測結果を追加する
val_result_df = val_df.copy()

val_result_df["predicted_score"] = (
    val_predictions
)

val_result_df["absolute_error"] = np.abs(
    val_result_df["erythema_score"]
    - val_result_df["predicted_score"]
)

val_result_df["correct"] = (
    val_result_df["erythema_score"]
    == val_result_df["predicted_score"]
)


# 部位別の画像数・正解率・MAEを集計する
region_evaluation = (
    val_result_df
    .groupby("region")
    .agg(
        画像数=("image_path", "size"),
        正解率=("correct", "mean"),
        MAE=("absolute_error", "mean")
    )
    .reindex(
        ["head", "upper", "trunk", "lower"]
    )
)

region_evaluation[
    ["正解率", "MAE"]
] = region_evaluation[
    ["正解率", "MAE"]
].round(3)

display(region_evaluation)

In [ ]:
# 検証用データに、紅斑スコア0～3の予測確率を追加する
patient_region_result = val_df.copy()

patient_region_result["probability_0"] = val_probabilities[:, 0]
patient_region_result["probability_1"] = val_probabilities[:, 1]
patient_region_result["probability_2"] = val_probabilities[:, 2]
patient_region_result["probability_3"] = val_probabilities[:, 3]

# 同じ患者・同じ部位の複数画像について、
# スコア0～3の予測確率を平均する
patient_region_evaluation = (
    patient_region_result
    .groupby(
        ["patient_id", "region"],
        as_index=False
    )
    .agg(
        正解スコア=("erythema_score", "first"),
        画像数=("image_path", "size"),
        probability_0=("probability_0", "mean"),
        probability_1=("probability_1", "mean"),
        probability_2=("probability_2", "mean"),
        probability_3=("probability_3", "mean")
    )
)

# 平均予測確率が最も高いスコアを、
# 患者・部位単位の予測スコアとする
probability_columns = [
    "probability_0",
    "probability_1",
    "probability_2",
    "probability_3"
]

patient_region_evaluation["予測スコア"] = np.argmax(
    patient_region_evaluation[probability_columns].to_numpy(),
    axis=1
)

# 正解との誤差と、完全一致したかを計算する
patient_region_evaluation["絶対誤差"] = np.abs(
    patient_region_evaluation["正解スコア"]
    - patient_region_evaluation["予測スコア"]
)

patient_region_evaluation["正解"] = (
    patient_region_evaluation["正解スコア"]
    == patient_region_evaluation["予測スコア"]
)

display(patient_region_evaluation.head(20))

In [ ]:
# 評価指標を計算する機能を読み込む
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    cohen_kappa_score
)

# 患者・部位単位の正解スコアを取り出す
patient_region_true = (
    patient_region_evaluation["正解スコア"]
    .astype("int32")
    .to_numpy()
)

# 患者・部位単位の予測スコアを取り出す
patient_region_pred = (
    patient_region_evaluation["予測スコア"]
    .astype("int32")
    .to_numpy()
)

# 完全に一致した割合を計算する
patient_region_accuracy = accuracy_score(
    patient_region_true,
    patient_region_pred
)

# 正解と予測が平均で何点離れているか計算する
patient_region_mae = mean_absolute_error(
    patient_region_true,
    patient_region_pred
)

# 正解との差が1点以内だった割合を計算する
patient_region_within_one = np.mean(
    np.abs(
        patient_region_true - patient_region_pred
    ) <= 1
)

# スコア間の距離を考慮した一致度を計算する
patient_region_qwk = cohen_kappa_score(
    patient_region_true,
    patient_region_pred,
    weights="quadratic"
)

# 評価結果を表示する
print(
    "患者・部位単位の評価数：",
    len(patient_region_evaluation)
)

print(
    "患者・部位単位 Accuracy：",
    round(patient_region_accuracy, 3)
)

print(
    "患者・部位単位 MAE：",
    round(patient_region_mae, 3)
)

print(
    "患者・部位単位 1点以内正解率：",
    round(patient_region_within_one, 3)
)

print(
    "患者・部位単位 QWK：",
    round(patient_region_qwk, 3)
)

In [ ]:
# 混同行列を作成する機能を読み込む
from sklearn.metrics import confusion_matrix

# 患者・部位単位の混同行列を作る
patient_region_confusion = confusion_matrix(
    patient_region_true,
    patient_region_pred,
    labels=[0, 1, 2, 3]
)

# 見やすい表へ変換する
patient_region_confusion_df = pd.DataFrame(
    patient_region_confusion,
    index=[
        "正解0",
        "正解1",
        "正解2",
        "正解3"
    ],
    columns=[
        "予測0",
        "予測1",
        "予測2",
        "予測3"
    ]
)

display(patient_region_confusion_df)

In [ ]:
# 患者・部位単位の結果を、身体部位別に集計する
patient_region_by_region = (
    patient_region_evaluation
    .groupby("region")
    .agg(
        評価数=("patient_id", "size"),
        正解率=("正解", "mean"),
        MAE=("絶対誤差", "mean")
    )
    .reindex(
        ["head", "upper", "trunk", "lower"]
    )
)

# 表示する桁数をそろえる
patient_region_by_region[
    ["正解率", "MAE"]
] = patient_region_by_region[
    ["正解率", "MAE"]
].round(3)

display(patient_region_by_region)


In [ ]:
# 画像単位と患者・部位単位の評価結果を比較する
evaluation_comparison = pd.DataFrame(
    {
        "画像単位": [
            val_accuracy_score,
            val_mae,
            val_within_one,
            val_qwk
        ],
        "患者・部位単位": [
            patient_region_accuracy,
            patient_region_mae,
            patient_region_within_one,
            patient_region_qwk
        ]
    },
    index=[
        "Accuracy",
        "MAE",
        "1点以内正解率",
        "QWK"
    ]
).round(3)

display(evaluation_comparison)

In [ ]:
# 患者・部位単位で、正解と予測の件数を比較する
patient_region_distribution = pd.DataFrame({
    "正解数": (
        pd.Series(patient_region_true)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "予測数": (
        pd.Series(patient_region_pred)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

patient_region_distribution.index.name = "紅斑スコア"

display(patient_region_distribution)

In [ ]:
from sklearn.metrics import confusion_matrix

# 患者・部位単位の混同行列を作る
patient_region_confusion = confusion_matrix(
    patient_region_true,
    patient_region_pred,
    labels=[0, 1, 2, 3]
)

patient_region_confusion_df = pd.DataFrame(
    patient_region_confusion,
    index=[
        "正解0",
        "正解1",
        "正解2",
        "正解3"
    ],
    columns=[
        "予測0",
        "予測1",
        "予測2",
        "予測3"
    ]
)

display(patient_region_confusion_df)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        patient_region_true,
        patient_region_pred,
        labels=[0, 1, 2, 3],
        target_names=[
            "紅斑0",
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 学習用データに存在する紅斑スコア
classes = np.array([0, 1, 2, 3])

# 紅斑スコアの少ないクラスほど大きな重みを計算する
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_model_df["erythema_score"].to_numpy()
)

# model.fit()で使える辞書形式にする
class_weights = {
    int(class_label): float(weight)
    for class_label, weight
    in zip(classes, class_weights_array)
}

print("クラス重み：", class_weights)

In [ ]:
display(
    train_model_df["erythema_score"]
    .value_counts()
    .reindex([0, 1, 2, 3], fill_value=0)
    .rename_axis("紅斑スコア")
    .reset_index(name="学習画像数")
)

In [ ]:
# 2と3のみ予想→画像に偏り？
# 学習用データの紅斑スコア別画像数を確認する
train_score_counts = (
    train_model_df["erythema_score"]
    .value_counts()
    .reindex([0, 1, 2, 3], fill_value=0)
    .sort_index()
)

train_score_counts.index.name = "紅斑スコア"

display(
    train_score_counts.to_frame(
        name="学習画像数"
    )
)

In [ ]:
import numpy as np

# 学習用画像の総数
total_train_images = len(train_model_df)

# 紅斑スコアのクラス数
num_classes = 4

# 頻度の少ないスコアほど大きな重みを付ける
# 平方根を使い、重みが極端になりすぎないようにする
class_weights = {}

for score in range(num_classes):
    count = train_score_counts.loc[score]

    if count == 0:
        raise ValueError(
            f"学習用データに紅斑スコア{score}がありません。"
        )

    balanced_weight = (
        total_train_images
        / (num_classes * count)
    )

    class_weights[score] = float(
        np.sqrt(balanced_weight)
    )

print("使用するクラス重み：")
print(class_weights)


In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# 重み付き学習用に、MobileNetV2モデルを新しく作り直す

base_model_weighted = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

# 第1段階ではMobileNetV2本体を固定する
base_model_weighted.trainable = False


# 入力層
inputs = layers.Input(
    shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

# データ拡張
x = data_augmentation(inputs)

# MobileNetV2用の前処理
x = preprocess_input(x)

# 画像特徴を抽出する
x = base_model_weighted(
    x,
    training=False
)

# 特徴を1次元へまとめる
x = layers.GlobalAveragePooling2D()(x)

# 過学習を抑える
x = layers.Dropout(0.3)(x)

# 紅斑スコア0～3の確率を出力する
outputs = layers.Dense(
    4,
    activation="softmax"
)(x)

# 重み付き学習用モデル
weighted_model = models.Model(
    inputs,
    outputs
)

# 学習方法を設定する
weighted_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

In [ ]:
# 重み付きモデルの保存先
weighted_weights_path = (
    model_dir
    / "erythema_mobilenetv2_stage1_weighted.weights.h5"
)

weighted_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(weighted_weights_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
# クラス重みを付けて、第1段階の学習を行う
history_weighted = weighted_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=weighted_callbacks,
    class_weight=class_weights,
    verbose=1
)

In [ ]:
# 重み付きモデルで検証用データを予測する
weighted_val_probabilities = weighted_model.predict(
    val_ds,
    verbose=1
)

weighted_val_predictions = np.argmax(
    weighted_val_probabilities,
    axis=1
)

# 正解数と予測数を比較する
weighted_distribution = pd.DataFrame({
    "正解数": (
        pd.Series(val_true)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "重みなし予測数": (
        pd.Series(val_predictions)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "重み付き予測数": (
        pd.Series(weighted_val_predictions)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

weighted_distribution.index.name = "紅斑スコア"

display(weighted_distribution)

In [ ]:
#評価指標の比較
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    cohen_kappa_score,
    classification_report
)

# 重み付きモデルのAccuracy
weighted_accuracy = accuracy_score(
    val_true,
    weighted_val_predictions
)

# 重み付きモデルのMAE
weighted_mae = mean_absolute_error(
    val_true,
    weighted_val_predictions
)

# 重み付きモデルの1点以内正解率
weighted_within_one = np.mean(
    np.abs(
        val_true - weighted_val_predictions
    ) <= 1
)

# 重み付きモデルのQWK
weighted_qwk = cohen_kappa_score(
    val_true,
    weighted_val_predictions,
    weights="quadratic"
)

# 重みなしモデルと重み付きモデルを比較する
weighted_evaluation_comparison = pd.DataFrame(
    {
        "重みなしモデル": [
            val_accuracy_score,
            val_mae,
            val_within_one,
            val_qwk
        ],
        "重み付きモデル": [
            weighted_accuracy,
            weighted_mae,
            weighted_within_one,
            weighted_qwk
        ]
    },
    index=[
        "Accuracy",
        "MAE",
        "1点以内正解率",
        "QWK"
    ]
).round(3)

display(weighted_evaluation_comparison)

In [ ]:
# 重み付きモデルの混合行列
from sklearn.metrics import confusion_matrix

weighted_confusion = confusion_matrix(
    val_true,
    weighted_val_predictions,
    labels=[0, 1, 2, 3]
)

weighted_confusion_df = pd.DataFrame(
    weighted_confusion,
    index=[
        "正解0",
        "正解1",
        "正解2",
        "正解3"
    ],
    columns=[
        "予測0",
        "予測1",
        "予測2",
        "予測3"
    ]
)

display(weighted_confusion_df)

In [ ]:
# クラス別評価

print(
    classification_report(
        val_true,
        weighted_val_predictions,
        labels=[0, 1, 2, 3],
        target_names=[
            "紅斑0",
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

 重み付きモデルでは予想分布は改善したが総合的な性能悪化→重みなしモデルをベースに
　　問題は検証データの正解数が偏っていること（０は一件しかない）

#### クラス重みによって予測分布の偏りは軽減したが、Accuracy、MAE、1点以内正解率、QWKはいずれも改善しなかったため、重みなしモデルをベースラインとして採用する。

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit


# 分割比率
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# 比較する紅斑スコア
SCORE_CLASSES = [0, 1, 2, 3]


def get_score_ratio(dataframe):
    """紅斑スコア0～3の画像割合を取得する"""

    return (
        dataframe["erythema_score"]
        .value_counts(normalize=True)
        .reindex(SCORE_CLASSES, fill_value=0)
        .to_numpy()
    )


# データ全体のスコア割合
overall_score_ratio = get_score_ratio(
    erythema_df
)

# データ全体の患者数と画像数
total_patients = erythema_df["patient_id"].nunique()
total_images = len(erythema_df)

split_results = []


# random_stateを変えながら500通りの患者単位分割を試す
for seed in range(500):

    # まず学習用70%と一時データ30%へ分割する
    first_split = GroupShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=seed
    )

    train_index, temp_index = next(
        first_split.split(
            erythema_df,
            groups=erythema_df["patient_id"]
        )
    )

    train_candidate = erythema_df.iloc[
        train_index
    ]

    temp_candidate = erythema_df.iloc[
        temp_index
    ]


    # 一時データを検証用とテスト用へ半分ずつ分ける
    second_split = GroupShuffleSplit(
        n_splits=1,
        test_size=0.50,
        random_state=seed
    )

    val_index, test_index = next(
        second_split.split(
            temp_candidate,
            groups=temp_candidate["patient_id"]
        )
    )

    val_candidate = temp_candidate.iloc[
        val_index
    ]

    test_candidate = temp_candidate.iloc[
        test_index
    ]


    # 各データ群のスコア別画像数
    train_counts = (
        train_candidate["erythema_score"]
        .value_counts()
        .reindex(SCORE_CLASSES, fill_value=0)
    )

    val_counts = (
        val_candidate["erythema_score"]
        .value_counts()
        .reindex(SCORE_CLASSES, fill_value=0)
    )

    test_counts = (
        test_candidate["erythema_score"]
        .value_counts()
        .reindex(SCORE_CLASSES, fill_value=0)
    )


    # 学習・検証・テストのいずれかで、
    # 正解スコアが完全に欠ける分割は除外する
    if (
        (train_counts == 0).any()
        or (val_counts == 0).any()
        or (test_counts == 0).any()
    ):
        continue


    # 全体のスコア割合との差を計算する
    score_difference = (
        np.abs(
            get_score_ratio(train_candidate)
            - overall_score_ratio
        ).sum()
        +
        np.abs(
            get_score_ratio(val_candidate)
            - overall_score_ratio
        ).sum()
        +
        np.abs(
            get_score_ratio(test_candidate)
            - overall_score_ratio
        ).sum()
    )


    # 各群の画像数比率と目標比率との差
    image_ratio_difference = (
        abs(
            len(train_candidate) / total_images
            - TRAIN_RATIO
        )
        +
        abs(
            len(val_candidate) / total_images
            - VAL_RATIO
        )
        +
        abs(
            len(test_candidate) / total_images
            - TEST_RATIO
        )
    )


    # 検証用・テスト用の少数クラス件数を補助評価する
    # 少数クラスが多い分割をやや優先する
    minority_bonus = (
        val_counts.loc[0]
        + val_counts.loc[1]
        + val_counts.loc[3]
        + test_counts.loc[0]
        + test_counts.loc[1]
        + test_counts.loc[3]
    )


    # 小さいほど偏りが少ない
    split_score = (
        score_difference
        + image_ratio_difference
        - 0.0001 * minority_bonus
    )


    split_results.append({
        "seed": seed,
        "split_score": split_score,
        "train_images": len(train_candidate),
        "val_images": len(val_candidate),
        "test_images": len(test_candidate),
        "train_patients": (
            train_candidate["patient_id"].nunique()
        ),
        "val_patients": (
            val_candidate["patient_id"].nunique()
        ),
        "test_patients": (
            test_candidate["patient_id"].nunique()
        ),
        "val_score_0": int(val_counts.loc[0]),
        "val_score_1": int(val_counts.loc[1]),
        "val_score_2": int(val_counts.loc[2]),
        "val_score_3": int(val_counts.loc[3]),
        "test_score_0": int(test_counts.loc[0]),
        "test_score_1": int(test_counts.loc[1]),
        "test_score_2": int(test_counts.loc[2]),
        "test_score_3": int(test_counts.loc[3])
    })


# 条件を満たす候補があるか確認する
if not split_results:
    raise ValueError(
        "検証用・テスト用の両方に"
        "紅斑スコア0～3が入る分割を作れませんでした。"
        "全体のスコア別件数を確認してください。"
    )


# 偏りが少ない順に並べる
split_search_df = (
    pd.DataFrame(split_results)
    .sort_values(
        "split_score",
        ascending=True
    )
    .reset_index(drop=True)
)

print(
    "条件を満たした分割候補数：",
    len(split_search_df)
)

print("【偏りが少ない分割候補 上位10件】")

display(
    split_search_df.head(10)
)

In [ ]:
# 最も偏りが少なかったrandom_stateを使用する
BEST_SEED = int(
    split_search_df.loc[0, "seed"]
)

print(
    "採用するrandom_state：",
    BEST_SEED
)


# 患者単位で学習用70%と一時データ30%へ分ける
first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=BEST_SEED
)

train_index, temp_index = next(
    first_split.split(
        erythema_df,
        groups=erythema_df["patient_id"]
    )
)

train_df = erythema_df.iloc[
    train_index
].copy()

temp_df = erythema_df.iloc[
    temp_index
].copy()


# 一時データを検証用15%とテスト用15%へ分ける
second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=BEST_SEED
)

val_index, test_index = next(
    second_split.split(
        temp_df,
        groups=temp_df["patient_id"]
    )
)

val_df = temp_df.iloc[
    val_index
].copy()

test_df = temp_df.iloc[
    test_index
].copy()


# 行番号を振り直す
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
# 各データ群の患者IDを集合にする
train_patients = set(
    train_df["patient_id"]
)

val_patients = set(
    val_df["patient_id"]
)

test_patients = set(
    test_df["patient_id"]
)


# 同じ患者が複数のデータ群にいないことを確認する
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

print("患者単位で重複なく分割できています。")


# 分割後の患者数と画像数
print(
    "学習用：",
    len(train_df), "画像 /",
    len(train_patients), "患者"
)

print(
    "検証用：",
    len(val_df), "画像 /",
    len(val_patients), "患者"
)

print(
    "テスト用：",
    len(test_df), "画像 /",
    len(test_patients), "患者"
)


# 分割後のスコア別画像数
new_split_distribution = pd.DataFrame({
    "全体": (
        erythema_df["erythema_score"]
        .value_counts()
        .reindex(SCORE_CLASSES, fill_value=0)
    ),
    "学習用": (
        train_df["erythema_score"]
        .value_counts()
        .reindex(SCORE_CLASSES, fill_value=0)
    ),
    "検証用": (
        val_df["erythema_score"]
        .value_counts()
        .reindex(SCORE_CLASSES, fill_value=0)
    ),
    "テスト用": (
        test_df["erythema_score"]
        .value_counts()
        .reindex(SCORE_CLASSES, fill_value=0)
    )
})

new_split_distribution.index.name = "紅斑スコア"

display(new_split_distribution)

In [ ]:
# 学習用だけ、同じ患者・同じ部位から
# 使用する画像を最大10枚に制限する
MAX_IMAGES_PER_PATIENT_REGION = 10

train_model_df = (
    train_df
    .groupby(
        ["patient_id", "region"],
        group_keys=False
    )
    .apply(
        lambda group: group.sample(
            n=min(
                len(group),
                MAX_IMAGES_PER_PATIENT_REGION
            ),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print(
    "枚数制限前の学習画像数：",
    len(train_df)
)

print(
    "枚数制限後の学習画像数：",
    len(train_model_df)
)

In [ ]:
# 患者・部位あたりの画像数が最大10枚になっていることを確認する
train_group_counts = (
    train_model_df
    .groupby(["patient_id", "region"])
    .size()
)

print(
    "患者・部位あたりの最大画像数：",
    train_group_counts.max()
)

assert train_group_counts.max() <= 10

print("学習用画像を最大10枚に制限できています。")

In [ ]:
# 新しい分割からTensorFlowデータセットを作り直す

IMG_SIZE = 224
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    """JPEG画像と紅斑スコアをモデル入力用に整える"""

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = tf.cast(
        image,
        tf.float32
    )

    label = tf.cast(
        label,
        tf.int32
    )

    return image, label


# 新しい分割から画像パスと紅斑スコアを取り出す
train_paths = (
    train_model_df["image_path"]
    .astype(str)
    .to_numpy()
)

train_labels = (
    train_model_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)

val_paths = (
    val_df["image_path"]
    .astype(str)
    .to_numpy()
)

val_labels = (
    val_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)

test_paths = (
    test_df["image_path"]
    .astype(str)
    .to_numpy()
)

test_labels = (
    test_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)


# TensorFlowデータセットを作る
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_paths, test_labels)
)


# 学習用データセット
train_ds = (
    train_ds
    .shuffle(
        buffer_size=len(train_model_df),
        seed=42,
        reshuffle_each_iteration=True
    )
    .map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


# 検証用データセット
val_ds = (
    val_ds
    .map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


# テスト用データセット
test_ds = (
    test_ds
    .map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
# 1バッチを正常に読み込めるか確認する
for image_batch, label_batch in train_ds.take(1):
    print("画像バッチの形：", image_batch.shape)
    print("ラベルバッチの形：", label_batch.shape)
    print("紅斑スコア：", label_batch.numpy())

In [ ]:
balanced_weights_path = (
    model_dir
    / "erythema_mobilenetv2_balanced_split.weights.h5"
)

In [ ]:
# モデル作成に必要な機能を読み込む
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


# 学習画像へランダムな変形を加える
balanced_data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ],
    name="balanced_data_augmentation"
)


# ImageNetで学習済みのMobileNetV2を新しく読み込む
balanced_base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

# 第1段階ではMobileNetV2本体を固定する
balanced_base_model.trainable = False


# モデルへ入力する画像の形を定義する
inputs = layers.Input(
    shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    ),
    name="input_image"
)

# 学習時だけデータ拡張を適用する
x = balanced_data_augmentation(inputs)

# MobileNetV2向けに画素値を変換する
x = preprocess_input(x)

# MobileNetV2で画像特徴を抽出する
x = balanced_base_model(
    x,
    training=False
)

# 特徴マップを1次元へまとめる
x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

# 過学習を抑える
x = layers.Dropout(
    0.3,
    name="dropout"
)(x)

# 紅斑スコア0～3の確率を出力する
outputs = layers.Dense(
    4,
    activation="softmax",
    name="erythema_output"
)(x)


# 新しい分割用モデルを作成する
balanced_model = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="erythema_mobilenetv2_balanced_split"
)


# モデルの学習方法を設定する
balanced_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)


# モデル構造を確認する
balanced_model.summary()

In [ ]:
# 新しい分割で学習した重みの保存場所
balanced_weights_path = (
    model_dir
    / "erythema_mobilenetv2_balanced_split.weights.h5"
)

print(
    "新しい重みの保存場所：",
    balanced_weights_path
)

In [ ]:
# 新しい分割用の学習停止・保存・学習率調整を設定する
balanced_callbacks = [
    # 検証損失が改善しなければ学習を終了する
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    # 検証損失が最小の重みを保存する
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(balanced_weights_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    # 検証損失が改善しなければ学習率を半分にする
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
# 新しい患者分割を使って、
# MobileNetV2本体を固定した第1段階の学習を行う
INITIAL_EPOCHS = 20

history_balanced = balanced_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=balanced_callbacks,
    verbose=1
)

In [ ]:
# 新しいモデルで検証用画像を予測する
balanced_val_probabilities = balanced_model.predict(
    val_ds,
    verbose=1
)

# 最も確率が高いスコアを予測結果とする
balanced_val_predictions = np.argmax(
    balanced_val_probabilities,
    axis=1
)

# 新しい検証用データの正解スコア
balanced_val_true = (
    val_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)

print(
    "検証画像数：",
    len(balanced_val_true)
)

print(
    "予測結果数：",
    len(balanced_val_predictions)
)

assert (
    len(balanced_val_true)
    == len(balanced_val_predictions)
)


In [ ]:
# 新しい分割での正解数と予測数を比較する
balanced_prediction_distribution = pd.DataFrame({
    "正解数": (
        pd.Series(balanced_val_true)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "予測数": (
        pd.Series(balanced_val_predictions)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

balanced_prediction_distribution.index.name = (
    "紅斑スコア"
)

display(balanced_prediction_distribution)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    cohen_kappa_score
)


# 完全一致率
balanced_accuracy = accuracy_score(
    balanced_val_true,
    balanced_val_predictions
)

# 平均絶対誤差
balanced_mae = mean_absolute_error(
    balanced_val_true,
    balanced_val_predictions
)

# 正解との差が1点以内の割合
balanced_within_one = np.mean(
    np.abs(
        balanced_val_true
        - balanced_val_predictions
    ) <= 1
)

# スコア間の距離を考慮した一致度
balanced_qwk = cohen_kappa_score(
    balanced_val_true,
    balanced_val_predictions,
    weights="quadratic"
)


print(
    "新しい分割 Accuracy：",
    round(balanced_accuracy, 3)
)

print(
    "新しい分割 MAE：",
    round(balanced_mae, 3)
)

print(
    "新しい分割 1点以内正解率：",
    round(balanced_within_one, 3)
)

print(
    "新しい分割 QWK：",
    round(balanced_qwk, 3)
)

In [ ]:
# 新しい検証データで、正解数と予測数を比較する
balanced_distribution = pd.DataFrame({
    "正解数": (
        pd.Series(balanced_val_true)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "予測数": (
        pd.Series(balanced_val_predictions)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

balanced_distribution.index.name = "紅斑スコア"

display(balanced_distribution)

In [ ]:
# 検証データで最も多い正解スコア
majority_class = (
    pd.Series(balanced_val_true)
    .value_counts()
    .idxmax()
)

# 全画像を多数派スコアと予測する
majority_predictions = np.full_like(
    balanced_val_true,
    fill_value=majority_class
)

# 多数派だけを予測した場合のAccuracyとMAE
majority_accuracy = accuracy_score(
    balanced_val_true,
    majority_predictions
)

majority_mae = mean_absolute_error(
    balanced_val_true,
    majority_predictions
)

print("検証データの多数派スコア：", majority_class)
print("多数派予測のAccuracy：", round(majority_accuracy, 3))
print("多数派予測のMAE：", round(majority_mae, 3))

print("\nMobileNetV2のAccuracy：", round(balanced_accuracy, 3))
print("MobileNetV2のMAE：", round(balanced_mae, 3))

In [ ]:
from sklearn.metrics import confusion_matrix

balanced_confusion = confusion_matrix(
    balanced_val_true,
    balanced_val_predictions,
    labels=[0, 1, 2, 3]
)

balanced_confusion_df = pd.DataFrame(
    balanced_confusion,
    index=["正解0", "正解1", "正解2", "正解3"],
    columns=["予測0", "予測1", "予測2", "予測3"]
)

display(balanced_confusion_df)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    confusion_matrix,
    classification_report
)

# 1. 正解数と予測数を比較する
score_distribution = pd.DataFrame({
    "正解数": (
        pd.Series(balanced_val_true)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "予測数": (
        pd.Series(balanced_val_predictions)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

score_distribution.index.name = "紅斑スコア"

print("【正解数と予測数】")
display(score_distribution)


# 2. 混同行列を作る
# 行が正解、列が予測
confusion = confusion_matrix(
    balanced_val_true,
    balanced_val_predictions,
    labels=[0, 1, 2, 3]
)

confusion_df = pd.DataFrame(
    confusion,
    index=["正解0", "正解1", "正解2", "正解3"],
    columns=["予測0", "予測1", "予測2", "予測3"]
)

print("【混同行列】")
display(confusion_df)


# 3. スコアごとのprecision・recall・F1を確認する
print("【クラス別評価】")

print(
    classification_report(
        balanced_val_true,
        balanced_val_predictions,
        labels=[0, 1, 2, 3],
        target_names=[
            "紅斑0",
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)


# 4. 多数派スコアだけを答える単純モデルと比較する
majority_class = (
    pd.Series(balanced_val_true)
    .value_counts()
    .idxmax()
)

majority_predictions = np.full_like(
    balanced_val_true,
    fill_value=majority_class
)

majority_accuracy = accuracy_score(
    balanced_val_true,
    majority_predictions
)

majority_mae = mean_absolute_error(
    balanced_val_true,
    majority_predictions
)

print("【多数派予測との比較】")
print("検証データの多数派スコア：", majority_class)
print("多数派予測のAccuracy：", round(majority_accuracy, 3))
print("MobileNetV2のAccuracy：", round(balanced_accuracy, 3))
print("多数派予測のMAE：", round(majority_mae, 3))
print("MobileNetV2のMAE：", round(balanced_mae, 3))

In [ ]:
# 多数派予測とMobileNetV2の結果を見やすく比較する
baseline_comparison = pd.DataFrame(
    {
        "多数派予測": [
            majority_accuracy,
            majority_mae
        ],
        "MobileNetV2": [
            balanced_accuracy,
            balanced_mae
        ]
    },
    index=[
        "Accuracy",
        "MAE"
    ]
).round(3)

display(baseline_comparison)

In [ ]:
# MobileNetV2本体を学習可能にする
balanced_base_model.trainable = True

# 上位30層だけ学習し、それより前の層は固定する
fine_tune_at = len(
    balanced_base_model.layers
) - 30

for layer in balanced_base_model.layers[:fine_tune_at]:
    layer.trainable = False

# BatchNormalization層は少量データで不安定になりやすいため固定する
for layer in balanced_base_model.layers:
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False

print(
    "MobileNetV2の総レイヤー数：",
    len(balanced_base_model.layers)
)

print(
    "学習可能なレイヤー数：",
    sum(
        layer.trainable
        for layer in balanced_base_model.layers
    )
)

In [ ]:
# ファインチューニングでは、
# 学習済みの重みを壊さないよう小さい学習率を使用する
balanced_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

In [ ]:
# ファインチューニング後の最良重みを保存する場所
finetune_weights_path = (
    model_dir
    / "erythema_mobilenetv2_finetuned.weights.h5"
)

print(
    "ファインチューニング重みの保存場所：",
    finetune_weights_path
)

In [ ]:
# ファインチューニング用のコールバック
finetune_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(finetune_weights_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
# MobileNetV2の上位層をEASI画像へ適応させる
FINE_TUNE_EPOCHS = 15

history_finetune = balanced_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=finetune_callbacks,
    verbose=1
)

In [ ]:
# ファインチューニング後のモデルで検証用画像を予測する
finetune_val_probabilities = balanced_model.predict(
    val_ds,
    verbose=1
)

finetune_val_predictions = np.argmax(
    finetune_val_probabilities,
    axis=1
)

In [ ]:
# ファインチューニング後の評価指標を計算する
finetune_accuracy = accuracy_score(
    balanced_val_true,
    finetune_val_predictions
)

finetune_mae = mean_absolute_error(
    balanced_val_true,
    finetune_val_predictions
)

finetune_within_one = np.mean(
    np.abs(
        balanced_val_true
        - finetune_val_predictions
    ) <= 1
)

finetune_qwk = cohen_kappa_score(
    balanced_val_true,
    finetune_val_predictions,
    weights="quadratic"
)


# ファインチューニング前後を比較する
finetune_comparison = pd.DataFrame(
    {
        "ファインチューニング前": [
            balanced_accuracy,
            balanced_mae,
            balanced_within_one,
            balanced_qwk
        ],
        "ファインチューニング後": [
            finetune_accuracy,
            finetune_mae,
            finetune_within_one,
            finetune_qwk
        ]
    },
    index=[
        "Accuracy",
        "MAE",
        "1点以内正解率",
        "QWK"
    ]
).round(3)

display(finetune_comparison)

In [ ]:
# ファインチューニング後の正解数と予測数を比較する
finetune_distribution = pd.DataFrame({
    "正解数": (
        pd.Series(balanced_val_true)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "ファインチューニング前": (
        pd.Series(balanced_val_predictions)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "ファインチューニング後": (
        pd.Series(finetune_val_predictions)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

finetune_distribution.index.name = "紅斑スコア"

print("【正解数と予測数】")
display(finetune_distribution)

In [ ]:
# ファインチューニング後の混同行列を作る
finetune_confusion = confusion_matrix(
    balanced_val_true,
    finetune_val_predictions,
    labels=[0, 1, 2, 3]
)

finetune_confusion_df = pd.DataFrame(
    finetune_confusion,
    index=[
        "正解0",
        "正解1",
        "正解2",
        "正解3"
    ],
    columns=[
        "予測0",
        "予測1",
        "予測2",
        "予測3"
    ]
)

print("【ファインチューニング後の混同行列】")
display(finetune_confusion_df)

In [ ]:
# ファインチューニング後のクラス別評価を表示する
print("【ファインチューニング後のクラス別評価】")

print(
    classification_report(
        balanced_val_true,
        finetune_val_predictions,
        labels=[0, 1, 2, 3],
        target_names=[
            "紅斑0",
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
###ファインチューニングする前のほうが精度が良い

In [ ]:
# ファインチューニング前の最良重みへ戻す
balanced_model.load_weights(
    str(balanced_weights_path)
)

print("ファインチューニング前の重みを読み込みました。")

In [ ]:
# 重みを戻したモデルで検証用データを予測する
restored_probabilities = balanced_model.predict(
    val_ds,
    verbose=1
)

restored_predictions = np.argmax(
    restored_probabilities,
    axis=1
)

# 評価指標を確認する
restored_accuracy = accuracy_score(
    balanced_val_true,
    restored_predictions
)

restored_mae = mean_absolute_error(
    balanced_val_true,
    restored_predictions
)

restored_qwk = cohen_kappa_score(
    balanced_val_true,
    restored_predictions,
    weights="quadratic"
)

print("復元後 Accuracy：", round(restored_accuracy, 3))
print("復元後 MAE：", round(restored_mae, 3))
print("復元後 QWK：", round(restored_qwk, 3))

#### 順序分類

In [ ]:
# 紅斑スコア0～3を、3つの順序ラベルへ変換する関数
def make_ordinal_label(score):
    """
    紅斑スコアを順序ラベルへ変換する

    0 → [0, 0, 0]
    1 → [1, 0, 0]
    2 → [1, 1, 0]
    3 → [1, 1, 1]
    """

    score = tf.cast(score, tf.int32)

    ordinal_label = tf.cast(
        score > tf.range(3),
        tf.float32
    )

    return ordinal_label

In [ ]:
# スコア0～3の変換結果を確認する
for score in range(4):
    ordinal_label = make_ordinal_label(score)

    print(
        f"紅斑スコア {score} →",
        ordinal_label.numpy()
    )

In [ ]:
# JPEG画像を読み込み、
# 紅斑スコアを順序ラベルへ変換する
def load_image_ordinal(path, score):
    """画像と順序ラベルをモデル入力用に整える"""

    # JPEG画像を読み込む
    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    # 画像サイズを224×224へそろえる
    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    # 画像を小数型へ変換する
    image = tf.cast(
        image,
        tf.float32
    )

    # 紅斑スコア0～3を順序ラベルへ変換する
    ordinal_label = make_ordinal_label(
        score
    )

    return image, ordinal_label

In [ ]:
# 画像パスと元の紅斑スコアから、
# 順序分類用TensorFlowデータセットを作る
train_ordinal_ds = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
)

val_ordinal_ds = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
)

test_ordinal_ds = tf.data.Dataset.from_tensor_slices(
    (test_paths, test_labels)
)

In [ ]:
# 順序分類の学習用データセット
train_ordinal_ds = (
    train_ordinal_ds
    .shuffle(
        buffer_size=len(train_model_df),
        seed=42,
        reshuffle_each_iteration=True
    )
    .map(
        load_image_ordinal,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
# 順序分類の検証用データセット
val_ordinal_ds = (
    val_ordinal_ds
    .map(
        load_image_ordinal,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


# 順序分類のテスト用データセット
test_ordinal_ds = (
    test_ordinal_ds
    .map(
        load_image_ordinal,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
# 順序分類用データセットを正常に作成できたか確認する
for image_batch, ordinal_label_batch in train_ordinal_ds.take(1):

    print(
        "画像バッチの形：",
        image_batch.shape
    )

    print(
        "順序ラベルの形：",
        ordinal_label_batch.shape
    )

    print(
        "順序ラベルの先頭10件："
    )

    print(
        ordinal_label_batch[:10].numpy()
    )

In [ ]:
# 順序分類用MobileNetV2モデルを作成する

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


# 学習画像に加えるデータ拡張を定義する
ordinal_data_augmentation = tf.keras.Sequential(
    [
        # 画像をランダムに左右反転する
        layers.RandomFlip("horizontal"),

        # 画像を小さい範囲でランダムに回転する
        layers.RandomRotation(0.05),

        # 画像を最大10%ランダムに拡大・縮小する
        layers.RandomZoom(0.10),

        # コントラストを小さい範囲で変更する
        layers.RandomContrast(0.10),
    ],
    name="ordinal_data_augmentation"
)


# ImageNetで学習済みのMobileNetV2を新しく読み込む
ordinal_base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

# 最初はMobileNetV2本体を固定する
ordinal_base_model.trainable = False


# モデルに入力する画像の形を定義する
ordinal_inputs = layers.Input(
    shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    ),
    name="input_image"
)

# 学習時だけデータ拡張を適用する
x = ordinal_data_augmentation(
    ordinal_inputs
)

# MobileNetV2用に画素値を変換する
x = preprocess_input(x)

# MobileNetV2を特徴抽出器として使う
x = ordinal_base_model(
    x,
    training=False
)

# 特徴マップを1次元にまとめる
x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

# 過学習を抑える
x = layers.Dropout(
    0.3,
    name="dropout"
)(x)

# 3つの境界を超える確率を出力する
#
# 出力0：紅斑スコアが0より大きい確率
# 出力1：紅斑スコアが1より大きい確率
# 出力2：紅斑スコアが2より大きい確率
ordinal_outputs = layers.Dense(
    3,
    activation="sigmoid",
    name="ordinal_output"
)(x)


# 順序分類モデルを作成する
ordinal_model = models.Model(
    inputs=ordinal_inputs,
    outputs=ordinal_outputs,
    name="erythema_mobilenetv2_ordinal"
)


# 順序分類モデルの学習方法を設定する
ordinal_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy"
        )
    ]
)


# モデル構造を表示する
ordinal_model.summary()

In [ ]:
# 順序分類モデルの最良重みを保存する場所
ordinal_weights_path = (
    model_dir
    / "erythema_mobilenetv2_ordinal.weights.h5"
)

print(
    "順序分類モデルの重み保存場所：",
    ordinal_weights_path
)


# 学習停止・最良重み保存・学習率調整を設定する
ordinal_callbacks = [
    # 検証損失が改善しなくなったら学習を終了する
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    # 検証損失が最小だった時点の重みを保存する
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ordinal_weights_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    # 検証損失が改善しなくなったら学習率を半分にする
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
# MobileNetV2本体を固定した状態で、
# 順序分類用の出力部分を学習する
ORDINAL_EPOCHS = 20

history_ordinal = ordinal_model.fit(
    train_ordinal_ds,
    validation_data=val_ordinal_ds,
    epochs=ORDINAL_EPOCHS,
    callbacks=ordinal_callbacks,
    verbose=1
)

In [ ]:
# 順序分類モデルの学習結果を確認する
print(
    "実際に学習したエポック数：",
    len(history_ordinal.history["loss"])
)

best_epoch = (
    np.argmin(history_ordinal.history["val_loss"])
    + 1
)

print(
    "検証損失が最小だったエポック：",
    best_epoch
)

print(
    "最小の検証損失：",
    round(
        min(history_ordinal.history["val_loss"]),
        4
    )
)

print(
    "重みファイルの存在：",
    ordinal_weights_path.is_file()
)


In [ ]:
# 学習済みの順序分類モデルで検証用画像を予測する
ordinal_val_probabilities = ordinal_model.predict(
    val_ordinal_ds,
    verbose=1
)

print(
    "予測確率の形：",
    ordinal_val_probabilities.shape
)

In [ ]:
# 0.5以上の境界確率の数を紅斑スコアとする
ORDINAL_THRESHOLD = 0.5

ordinal_val_predictions = (
    ordinal_val_probabilities
    >= ORDINAL_THRESHOLD
).sum(axis=1).astype("int32")

# 正解スコア
ordinal_val_true = (
    val_df["erythema_score"]
    .astype("int32")
    .to_numpy()
)

print("正解数：", len(ordinal_val_true))
print("予測数：", len(ordinal_val_predictions))

assert (
    len(ordinal_val_true)
    == len(ordinal_val_predictions)
)

In [ ]:
# 順序分類モデルの正解数と予測数を比較する
ordinal_distribution = pd.DataFrame({
    "正解数": (
        pd.Series(ordinal_val_true)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    ),
    "順序分類予測数": (
        pd.Series(ordinal_val_predictions)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )
})

ordinal_distribution.index.name = "紅斑スコア"

display(ordinal_distribution)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

# 完全一致率
ordinal_accuracy = accuracy_score(
    ordinal_val_true,
    ordinal_val_predictions
)

# 正解と予測の平均点差
ordinal_mae = mean_absolute_error(
    ordinal_val_true,
    ordinal_val_predictions
)

# 正解との差が1点以内だった割合
ordinal_within_one = np.mean(
    np.abs(
        ordinal_val_true - ordinal_val_predictions
    ) <= 1
)

# スコア間の距離を考慮した一致度
ordinal_qwk = cohen_kappa_score(
    ordinal_val_true,
    ordinal_val_predictions,
    weights="quadratic"
)

print("順序分類 Accuracy：", round(ordinal_accuracy, 3))
print("順序分類 MAE：", round(ordinal_mae, 3))
print(
    "順序分類 1点以内正解率：",
    round(ordinal_within_one, 3)
)
print("順序分類 QWK：", round(ordinal_qwk, 3))

In [ ]:
# 同じ検証データで4クラス分類と順序分類を比較する
model_comparison = pd.DataFrame(
    {
        "4クラス分類": [
            balanced_accuracy,
            balanced_mae,
            balanced_within_one,
            balanced_qwk
        ],
        "順序分類": [
            ordinal_accuracy,
            ordinal_mae,
            ordinal_within_one,
            ordinal_qwk
        ]
    },
    index=[
        "Accuracy",
        "MAE",
        "1点以内正解率",
        "QWK"
    ]
).round(3)

display(model_comparison)

In [ ]:
# 順序分類モデルの混同行列
ordinal_confusion = confusion_matrix(
    ordinal_val_true,
    ordinal_val_predictions,
    labels=[0, 1, 2, 3]
)

ordinal_confusion_df = pd.DataFrame(
    ordinal_confusion,
    index=["正解0", "正解1", "正解2", "正解3"],
    columns=["予測0", "予測1", "予測2", "予測3"]
)

print("【順序分類の混同行列】")
display(ordinal_confusion_df)


# 順序分類モデルのクラス別評価
print("【順序分類のクラス別評価】")

print(
    classification_report(
        ordinal_val_true,
        ordinal_val_predictions,
        labels=[0, 1, 2, 3],
        target_names=[
            "紅斑0",
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
### 学習データが本当に画像モデルで学習可能な構造になっているかを確認

In [ ]:
# 患者・部位ごとに1行だけ残し、
# 独立した紅斑評価単位でスコア分布を確認する
patient_region_labels = (
    erythema_df[
        [
            "patient_id",
            "region",
            "erythema_score"
        ]
    ]
    .drop_duplicates(
        subset=["patient_id", "region"]
    )
    .reset_index(drop=True)
)

print(
    "患者・部位の組み合わせ数：",
    len(patient_region_labels)
)

patient_region_score_counts = (
    patient_region_labels["erythema_score"]
    .value_counts()
    .reindex([0, 1, 2, 3], fill_value=0)
    .sort_index()
)

patient_region_score_counts.index.name = "紅斑スコア"

display(
    patient_region_score_counts.to_frame(
        name="患者・部位数"
    )
)

In [ ]:
# 身体部位別・紅斑スコア別の患者数を確認する
patient_region_crosstab = pd.crosstab(
    patient_region_labels["region"],
    patient_region_labels["erythema_score"]
).reindex(
    index=["head", "upper", "trunk", "lower"],
    columns=[0, 1, 2, 3],
    fill_value=0
)

display(patient_region_crosstab)

In [ ]:
### 紅斑0は患者・部位単位で6件のみであり、上肢および下肢には症例が存在しなかった。このため、4段階分類では多数クラスである紅斑2への予測集中が認められ、紅斑0の評価は困難であった。

In [ ]:
# 紅斑スコア1～3のみを対象とする補助解析用データ
erythema_123_df = (
    erythema_df[
        erythema_df["erythema_score"].isin([1, 2, 3])
    ]
    .copy()
    .reset_index(drop=True)
)

print("対象画像数：", len(erythema_123_df))
print(
    "対象患者数：",
    erythema_123_df["patient_id"].nunique()
)

# 患者・部位単位のスコア分布
erythema_123_patient_region = (
    erythema_123_df[
        ["patient_id", "region", "erythema_score"]
    ]
    .drop_duplicates(
        subset=["patient_id", "region"]
    )
)

display(
    erythema_123_patient_region["erythema_score"]
    .value_counts()
    .reindex([1, 2, 3], fill_value=0)
    .rename_axis("紅斑スコア")
    .to_frame("患者・部位数")
)

In [ ]:
# 紅斑スコア1～3だけを対象とする
# 紅斑が認められる画像の重症度分類用データを作る
erythema_123_df = (
    erythema_df[
        erythema_df["erythema_score"].isin([1, 2, 3])
    ]
    .copy()
    .reset_index(drop=True)
)

# 元の紅斑スコアは残しておく
erythema_123_df["original_score"] = (
    erythema_123_df["erythema_score"]
    .astype("int32")
)

# モデル学習用に、1・2・3を0・1・2へ変換する
label_map = {
    1: 0,
    2: 1,
    3: 2
}

erythema_123_df["label"] = (
    erythema_123_df["original_score"]
    .map(label_map)
    .astype("int32")
)

# 必要な列を並べる
erythema_123_df = erythema_123_df[
    [
        "patient_id",
        "region",
        "image_path",
        "original_score",
        "label"
    ]
]

print(
    "紅斑1～3の対象画像数：",
    len(erythema_123_df)
)

print(
    "紅斑1～3の対象患者数：",
    erythema_123_df["patient_id"].nunique()
)

display(erythema_123_df.head(10))

In [ ]:
# 患者・部位ごとに1行だけ残す
patient_region_123 = (
    erythema_123_df[
        [
            "patient_id",
            "region",
            "original_score",
            "label"
        ]
    ]
    .drop_duplicates(
        subset=["patient_id", "region"]
    )
    .reset_index(drop=True)
)

print(
    "患者・部位の組み合わせ数：",
    len(patient_region_123)
)

# 元の紅斑スコア1～3の件数を表示する
score_123_counts = (
    patient_region_123["original_score"]
    .value_counts()
    .reindex([1, 2, 3], fill_value=0)
    .sort_index()
    .rename_axis("元の紅斑スコア")
    .reset_index(name="患者・部位数")
)

display(score_123_counts)


In [ ]:
# 部位別・紅斑スコア別の患者・部位数を確認する
region_score_123 = pd.crosstab(
    patient_region_123["region"],
    patient_region_123["original_score"]
).reindex(
    index=["head", "upper", "trunk", "lower"],
    columns=[1, 2, 3],
    fill_value=0
)

display(region_score_123)

In [ ]:
# 紅斑スコア1～3のデータを、
# 同じ患者が複数のデータ群に入らないよう患者単位で分割する

from sklearn.model_selection import GroupShuffleSplit


# まず学習用70%と一時データ30%に分ける
first_split_123 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_index_123, temp_index_123 = next(
    first_split_123.split(
        erythema_123_df,
        groups=erythema_123_df["patient_id"]
    )
)

train_123_df = (
    erythema_123_df
    .iloc[train_index_123]
    .copy()
    .reset_index(drop=True)
)

temp_123_df = (
    erythema_123_df
    .iloc[temp_index_123]
    .copy()
    .reset_index(drop=True)
)


# 一時データ30%を検証用15%とテスト用15%に分ける
second_split_123 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_index_123, test_index_123 = next(
    second_split_123.split(
        temp_123_df,
        groups=temp_123_df["patient_id"]
    )
)

val_123_df = (
    temp_123_df
    .iloc[val_index_123]
    .copy()
    .reset_index(drop=True)
)

test_123_df = (
    temp_123_df
    .iloc[test_index_123]
    .copy()
    .reset_index(drop=True)
)

In [ ]:
# 各データ群の患者IDを集合にする
train_123_patients = set(
    train_123_df["patient_id"]
)

val_123_patients = set(
    val_123_df["patient_id"]
)

test_123_patients = set(
    test_123_df["patient_id"]
)


# 同じ患者が複数のデータ群に入っていないことを確認する
assert train_123_patients.isdisjoint(
    val_123_patients
)

assert train_123_patients.isdisjoint(
    test_123_patients
)

assert val_123_patients.isdisjoint(
    test_123_patients
)

print("患者単位で重複なく分割できています。")

In [ ]:
# 学習・検証・テストの画像数と患者数を確認する
print(
    "学習用：",
    len(train_123_df),
    "画像 /",
    len(train_123_patients),
    "患者"
)

print(
    "検証用：",
    len(val_123_df),
    "画像 /",
    len(val_123_patients),
    "患者"
)

print(
    "テスト用：",
    len(test_123_df),
    "画像 /",
    len(test_123_patients),
    "患者"
)

In [ ]:
# 分割後の紅斑スコア別画像数を確認する
split_123_distribution = pd.DataFrame({
    "学習用": (
        train_123_df["original_score"]
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    ),
    "検証用": (
        val_123_df["original_score"]
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    ),
    "テスト用": (
        test_123_df["original_score"]
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    )
})

split_123_distribution.index.name = "紅斑スコア"

display(split_123_distribution)

In [ ]:
# 患者・部位ごとに1行だけ残して、
# 分割後の紅斑スコア分布を確認する
def patient_region_distribution(dataframe):

    patient_region = (
        dataframe[
            [
                "patient_id",
                "region",
                "original_score"
            ]
        ]
        .drop_duplicates(
            subset=["patient_id", "region"]
        )
    )

    return (
        patient_region["original_score"]
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    )


patient_region_split_123 = pd.DataFrame({
    "学習用": patient_region_distribution(
        train_123_df
    ),
    "検証用": patient_region_distribution(
        val_123_df
    ),
    "テスト用": patient_region_distribution(
        test_123_df
    )
})

patient_region_split_123.index.name = "紅斑スコア"

print("【患者・部位単位のスコア分布】")
display(patient_region_split_123)

In [ ]:
# 学習用データについて、
# 同じ患者・同じ部位から使用する画像を最大10枚に制限する
MAX_IMAGES_PER_PATIENT_REGION = 10

train_123_model_df = (
    train_123_df
    .groupby(
        ["patient_id", "region"],
        group_keys=False
    )
    .apply(
        lambda group: group.sample(
            n=min(
                len(group),
                MAX_IMAGES_PER_PATIENT_REGION
            ),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print(
    "枚数制限前の学習画像数：",
    len(train_123_df)
)

print(
    "枚数制限後の学習画像数：",
    len(train_123_model_df)
)

print(
    "学習用患者数：",
    train_123_model_df["patient_id"].nunique()
)

In [ ]:
# 患者・部位あたりの画像数を確認する
train_123_group_counts = (
    train_123_model_df
    .groupby(["patient_id", "region"])
    .size()
)

print(
    "制限後の患者・部位あたり最大画像数：",
    train_123_group_counts.max()
)

assert (
    train_123_group_counts.max()
    <= MAX_IMAGES_PER_PATIENT_REGION
)

print("患者・部位あたり最大10枚に制限できています。")

In [ ]:
# 枚数制限前後の紅斑スコア別画像数を比較する
score_123_comparison = pd.DataFrame({
    "制限前": (
        train_123_df["original_score"]
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    ),
    "制限後": (
        train_123_model_df["original_score"]
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    )
})

score_123_comparison.index.name = "紅斑スコア"

display(score_123_comparison)

In [ ]:
# 学習用の画像パスと3クラスラベルを取り出す
train_123_paths = (
    train_123_model_df["image_path"]
    .astype(str)
    .to_numpy()
)

train_123_labels = (
    train_123_model_df["label"]
    .astype("int32")
    .to_numpy()
)


# 検証用の画像パスと3クラスラベルを取り出す
val_123_paths = (
    val_123_df["image_path"]
    .astype(str)
    .to_numpy()
)

val_123_labels = (
    val_123_df["label"]
    .astype("int32")
    .to_numpy()
)


# テスト用の画像パスと3クラスラベルを取り出す
test_123_paths = (
    test_123_df["image_path"]
    .astype(str)
    .to_numpy()
)

test_123_labels = (
    test_123_df["label"]
    .astype("int32")
    .to_numpy()
)

In [ ]:
# 画像パスと3クラスラベルからTensorFlowデータセットを作る
train_123_ds = tf.data.Dataset.from_tensor_slices(
    (train_123_paths, train_123_labels)
)

val_123_ds = tf.data.Dataset.from_tensor_slices(
    (val_123_paths, val_123_labels)
)

test_123_ds = tf.data.Dataset.from_tensor_slices(
    (test_123_paths, test_123_labels)
)

In [ ]:
# 3クラス分類の学習用データセット
train_123_ds = (
    train_123_ds
    .shuffle(
        buffer_size=len(train_123_model_df),
        seed=42,
        reshuffle_each_iteration=True
    )
    .map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
# 3クラス分類の検証用データセット
val_123_ds = (
    val_123_ds
    .map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


# 3クラス分類のテスト用データセット
test_123_ds = (
    test_123_ds
    .map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
# 3クラス分類用データセットが正常に作成できたか確認する
for image_batch, label_batch in train_123_ds.take(1):

    print(
        "画像バッチの形：",
        image_batch.shape
    )

    print(
        "ラベルバッチの形：",
        label_batch.shape
    )

    print(
        "モデル内部のラベル：",
        label_batch.numpy()
    )

In [ ]:
# 紅斑スコア1・2・3を分類する
# 3クラスMobileNetV2モデルを作成する

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


# 学習画像へランダムな変形を加える
data_augmentation_123 = tf.keras.Sequential(
    [
        # 左右反転
        layers.RandomFlip("horizontal"),

        # 小さい範囲で回転
        layers.RandomRotation(0.05),

        # 最大10%の拡大・縮小
        layers.RandomZoom(0.10),

        # 小さい範囲でコントラストを変更
        layers.RandomContrast(0.10),
    ],
    name="data_augmentation_123"
)


# ImageNetで学習済みのMobileNetV2を読み込む
base_model_123 = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

# 最初はMobileNetV2本体を固定する
base_model_123.trainable = False


# 入力画像を定義する
inputs_123 = layers.Input(
    shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    ),
    name="input_image"
)

# 学習時だけデータ拡張を適用する
x = data_augmentation_123(inputs_123)

# MobileNetV2用に画素値を変換する
x = preprocess_input(x)

# MobileNetV2で画像特徴を抽出する
x = base_model_123(
    x,
    training=False
)

# 特徴マップを1次元へまとめる
x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

# 過学習を抑える
x = layers.Dropout(
    0.3,
    name="dropout"
)(x)

# 3クラスの予測確率を出力する
#
# 出力0：紅斑スコア1
# 出力1：紅斑スコア2
# 出力2：紅斑スコア3
outputs_123 = layers.Dense(
    3,
    activation="softmax",
    name="erythema_123_output"
)(x)


# 3クラス分類モデルを作る
model_123 = models.Model(
    inputs=inputs_123,
    outputs=outputs_123,
    name="erythema_mobilenetv2_123"
)


# 学習方法を設定する
model_123.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)


# モデル構造を確認する
model_123.summary()

In [ ]:
# 3クラスモデルの最良重みを保存する場所
weights_123_path = (
    model_dir
    / "erythema_mobilenetv2_123.weights.h5"
)

print(
    "3クラスモデルの重み保存場所：",
    weights_123_path
)

In [ ]:
# 3クラスモデル用のコールバックを設定する
callbacks_123 = [
    # 検証損失が改善しなければ学習を自動終了する
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    # 検証損失が最小だった重みを保存する
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(weights_123_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    # 検証損失が改善しなければ学習率を半分にする
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
# MobileNetV2本体を固定した状態で、
# 紅斑1・2・3の分類部分を学習する
EPOCHS_123 = 20

history_123 = model_123.fit(
    train_123_ds,
    validation_data=val_123_ds,
    epochs=EPOCHS_123,
    callbacks=callbacks_123,
    verbose=1
)

In [ ]:
# 実際に学習したエポック数を確認する
print(
    "実際に学習したエポック数：",
    len(history_123.history["loss"])
)

# 検証損失が最小だったエポックを確認する
best_epoch_123 = (
    np.argmin(
        history_123.history["val_loss"]
    )
    + 1
)

print(
    "検証損失が最小だったエポック：",
    best_epoch_123
)

print(
    "最小の検証損失：",
    round(
        min(history_123.history["val_loss"]),
        4
    )
)

print(
    "重みファイルの存在：",
    weights_123_path.is_file()
)

In [ ]:
# 3クラスモデルで検証用画像を予測する
val_123_probabilities = model_123.predict(
    val_123_ds,
    verbose=1
)

# 最も確率が高いモデル内部ラベル0・1・2を取得する
val_123_predictions = np.argmax(
    val_123_probabilities,
    axis=1
)

# 検証用データの正解ラベル0・1・2
val_123_true = (
    val_123_df["label"]
    .astype("int32")
    .to_numpy()
)

print("正解数：", len(val_123_true))
print("予測数：", len(val_123_predictions))

assert len(val_123_true) == len(val_123_predictions)

print("正解数と予測数が一致しています。")

In [ ]:
# モデル内部ラベル0・1・2を、元の紅斑スコア1・2・3へ戻す
val_123_true_scores = val_123_true + 1
val_123_predicted_scores = val_123_predictions + 1

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

accuracy_123 = accuracy_score(
    val_123_true_scores,
    val_123_predicted_scores
)

mae_123 = mean_absolute_error(
    val_123_true_scores,
    val_123_predicted_scores
)

within_one_123 = np.mean(
    np.abs(
        val_123_true_scores
        - val_123_predicted_scores
    ) <= 1
)

qwk_123 = cohen_kappa_score(
    val_123_true_scores,
    val_123_predicted_scores,
    weights="quadratic"
)

print("3クラス Accuracy：", round(accuracy_123, 3))
print("3クラス MAE：", round(mae_123, 3))
print("3クラス 1点以内正解率：", round(within_one_123, 3))
print("3クラス QWK：", round(qwk_123, 3))

In [ ]:
# 3クラスモデルの混同行列
confusion_123 = confusion_matrix(
    val_123_true_scores,
    val_123_predicted_scores,
    labels=[1, 2, 3]
)

confusion_123_df = pd.DataFrame(
    confusion_123,
    index=["正解1", "正解2", "正解3"],
    columns=["予測1", "予測2", "予測3"]
)

display(confusion_123_df)


# クラス別のprecision・recall・F1を表示する
print(
    classification_report(
        val_123_true_scores,
        val_123_predicted_scores,
        labels=[1, 2, 3],
        target_names=[
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# モデル内部の3クラス
classes_123 = np.array([0, 1, 2])

# 学習用データからクラス重みを計算する
class_weights_123_array = compute_class_weight(
    class_weight="balanced",
    classes=classes_123,
    y=train_123_model_df["label"].to_numpy()
)

# model.fitで使用できる辞書形式へ変換する
class_weights_123 = {
    int(class_label): float(weight)
    for class_label, weight
    in zip(
        classes_123,
        class_weights_123_array
    )
}

print("3クラス用のクラス重み：")
print(class_weights_123)

In [ ]:
# 重みが極端になりすぎないよう平方根を取る
class_weights_123_sqrt = {
    class_label: float(np.sqrt(weight))
    for class_label, weight
    in class_weights_123.items()
}

print("学習に使用する緩やかなクラス重み：")
print(class_weights_123_sqrt)

In [ ]:
# クラス重み付き3クラスモデルを新しく作る
weighted_base_model_123 = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

weighted_base_model_123.trainable = False

weighted_inputs_123 = layers.Input(
    shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

x = data_augmentation_123(
    weighted_inputs_123
)

x = preprocess_input(x)

x = weighted_base_model_123(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

weighted_outputs_123 = layers.Dense(
    3,
    activation="softmax"
)(x)

weighted_model_123 = models.Model(
    weighted_inputs_123,
    weighted_outputs_123,
    name="erythema_mobilenetv2_123_weighted"
)

weighted_model_123.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

In [ ]:
weighted_123_path = (
    model_dir
    / "erythema_mobilenetv2_123_weighted.weights.h5"
)

weighted_123_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(weighted_123_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
history_weighted_123 = weighted_model_123.fit(
    train_123_ds,
    validation_data=val_123_ds,
    epochs=20,
    callbacks=weighted_123_callbacks,
    class_weight=class_weights_123_sqrt,
    verbose=1
)


In [ ]:
# クラス重み付き3クラスモデルで検証画像を予測する
weighted_123_probabilities = weighted_model_123.predict(
    val_123_ds,
    verbose=1
)

# 最も確率が高い内部ラベル0・1・2を取得する
weighted_123_predictions = np.argmax(
    weighted_123_probabilities,
    axis=1
)

# 正解ラベル0・1・2を取得する
weighted_123_true = (
    val_123_df["label"]
    .astype("int32")
    .to_numpy()
)

print("正解数：", len(weighted_123_true))
print("予測数：", len(weighted_123_predictions))

assert (
    len(weighted_123_true)
    == len(weighted_123_predictions)
)

print("正解数と予測数が一致しています。")

In [ ]:
# 内部ラベル0・1・2を元の紅斑スコア1・2・3へ戻す
weighted_123_true_scores = (
    weighted_123_true + 1
)

weighted_123_predicted_scores = (
    weighted_123_predictions + 1
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

# 完全一致率
weighted_123_accuracy = accuracy_score(
    weighted_123_true_scores,
    weighted_123_predicted_scores
)

# 正解との平均点差
weighted_123_mae = mean_absolute_error(
    weighted_123_true_scores,
    weighted_123_predicted_scores
)

# 正解との差が1点以内だった割合
weighted_123_within_one = np.mean(
    np.abs(
        weighted_123_true_scores
        - weighted_123_predicted_scores
    ) <= 1
)

# スコアの順序と誤差の大きさを考慮した一致度
weighted_123_qwk = cohen_kappa_score(
    weighted_123_true_scores,
    weighted_123_predicted_scores,
    weights="quadratic"
)

print(
    "重み付き3クラス Accuracy：",
    round(weighted_123_accuracy, 3)
)

print(
    "重み付き3クラス MAE：",
    round(weighted_123_mae, 3)
)

print(
    "重み付き3クラス 1点以内正解率：",
    round(weighted_123_within_one, 3)
)

print(
    "重み付き3クラス QWK：",
    round(weighted_123_qwk, 3)
)

In [ ]:
# 重みなしと重み付きの3クラスモデルを比較する
comparison_123 = pd.DataFrame(
    {
        "重みなし3クラス": [
            accuracy_123,
            mae_123,
            within_one_123,
            qwk_123
        ],
        "重み付き3クラス": [
            weighted_123_accuracy,
            weighted_123_mae,
            weighted_123_within_one,
            weighted_123_qwk
        ]
    },
    index=[
        "Accuracy",
        "MAE",
        "1点以内正解率",
        "QWK"
    ]
).round(3)

display(comparison_123)

In [ ]:
# 予測が紅斑2へ集中していないか確認する
weighted_123_distribution = pd.DataFrame({
    "正解数": (
        pd.Series(weighted_123_true_scores)
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    ),
    "重みなし予測数": (
        pd.Series(val_123_predicted_scores)
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    ),
    "重み付き予測数": (
        pd.Series(weighted_123_predicted_scores)
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    )
})

weighted_123_distribution.index.name = "紅斑スコア"

display(weighted_123_distribution)

In [ ]:
# 重み付き3クラスモデルの混同行列を作る
weighted_123_confusion = confusion_matrix(
    weighted_123_true_scores,
    weighted_123_predicted_scores,
    labels=[1, 2, 3]
)

weighted_123_confusion_df = pd.DataFrame(
    weighted_123_confusion,
    index=[
        "正解1",
        "正解2",
        "正解3"
    ],
    columns=[
        "予測1",
        "予測2",
        "予測3"
    ]
)

display(weighted_123_confusion_df)

In [ ]:
# 紅斑1・2・3それぞれの性能を確認する
print(
    classification_report(
        weighted_123_true_scores,
        weighted_123_predicted_scores,
        labels=[1, 2, 3],
        target_names=[
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
### クラス重みなしモデルではAccuracy 0.648、MAE 0.354であったが、紅斑3のRecallは0であり、多数クラスである紅斑2への予測集中が認められた。クラス重み付きモデルではAccuracyは0.535、MAEは0.507へ低下した一方、Quadratic Weighted Kappaは0.058から0.123へ、Macro F1は0.305から0.386へ改善した。また、紅斑1および紅斑3のRecallはそれぞれ0.347、0.141となり、少数クラスを一定数識別できるようになった。

In [ ]:
# 検証データへ重み付きモデルの予測確率を追加する
weighted_123_result_df = val_123_df.copy()

weighted_123_result_df["probability_1"] = (
    weighted_123_probabilities[:, 0]
)

weighted_123_result_df["probability_2"] = (
    weighted_123_probabilities[:, 1]
)

weighted_123_result_df["probability_3"] = (
    weighted_123_probabilities[:, 2]
)


# 同じ患者・同じ部位の複数画像について、
# 各スコアの予測確率を平均する
weighted_123_patient_region = (
    weighted_123_result_df
    .groupby(
        ["patient_id", "region"],
        as_index=False
    )
    .agg(
        正解スコア=("original_score", "first"),
        画像数=("image_path", "size"),
        probability_1=("probability_1", "mean"),
        probability_2=("probability_2", "mean"),
        probability_3=("probability_3", "mean")
    )
)


# 平均確率が最も高いクラスを選び、
# 1を足して元の紅斑スコア1～3へ戻す
probability_columns_123 = [
    "probability_1",
    "probability_2",
    "probability_3"
]

weighted_123_patient_region["予測スコア"] = (
    np.argmax(
        weighted_123_patient_region[
            probability_columns_123
        ].to_numpy(),
        axis=1
    )
    + 1
)

display(weighted_123_patient_region.head(20))

In [ ]:
# 検証データに、重み付きモデルの予測確率を追加する
weighted_123_result_df = val_123_df.copy()

# モデル内部の出力0・1・2は、
# 元の紅斑スコア1・2・3に対応する
weighted_123_result_df["probability_1"] = (
    weighted_123_probabilities[:, 0]
)

weighted_123_result_df["probability_2"] = (
    weighted_123_probabilities[:, 1]
)

weighted_123_result_df["probability_3"] = (
    weighted_123_probabilities[:, 2]
)


# 同じ患者・同じ部位の複数画像について、
# 各紅斑スコアの予測確率を平均する
weighted_123_patient_region = (
    weighted_123_result_df
    .groupby(
        ["patient_id", "region"],
        as_index=False
    )
    .agg(
        正解スコア=("original_score", "first"),
        画像数=("image_path", "size"),
        probability_1=("probability_1", "mean"),
        probability_2=("probability_2", "mean"),
        probability_3=("probability_3", "mean")
    )
)


# 平均確率が最も高いクラスを選ぶ
probability_columns_123 = [
    "probability_1",
    "probability_2",
    "probability_3"
]

# argmaxの0・1・2に1を足し、
# 元の紅斑スコア1・2・3へ戻す
weighted_123_patient_region["予測スコア"] = (
    np.argmax(
        weighted_123_patient_region[
            probability_columns_123
        ].to_numpy(),
        axis=1
    )
    + 1
)


# 正解との点数差を計算する
weighted_123_patient_region["絶対誤差"] = np.abs(
    weighted_123_patient_region["正解スコア"]
    - weighted_123_patient_region["予測スコア"]
)


# 完全に一致したかを記録する
weighted_123_patient_region["正解"] = (
    weighted_123_patient_region["正解スコア"]
    == weighted_123_patient_region["予測スコア"]
)

display(
    weighted_123_patient_region.head(20)
)

In [ ]:
# 患者・部位単位の正解スコアと予測スコアを取り出す
patient_region_123_true = (
    weighted_123_patient_region["正解スコア"]
    .astype("int32")
    .to_numpy()
)

patient_region_123_pred = (
    weighted_123_patient_region["予測スコア"]
    .astype("int32")
    .to_numpy()
)


# 完全一致率
patient_region_123_accuracy = accuracy_score(
    patient_region_123_true,
    patient_region_123_pred
)

# 平均点差
patient_region_123_mae = mean_absolute_error(
    patient_region_123_true,
    patient_region_123_pred
)

# 正解との差が1点以内だった割合
patient_region_123_within_one = np.mean(
    np.abs(
        patient_region_123_true
        - patient_region_123_pred
    ) <= 1
)

# 順序と誤差の大きさを考慮した一致度
patient_region_123_qwk = cohen_kappa_score(
    patient_region_123_true,
    patient_region_123_pred,
    weights="quadratic"
)


print(
    "患者・部位単位の評価数：",
    len(patient_region_123_true)
)

print(
    "患者・部位単位 Accuracy：",
    round(patient_region_123_accuracy, 3)
)

print(
    "患者・部位単位 MAE：",
    round(patient_region_123_mae, 3)
)

print(
    "患者・部位単位 1点以内正解率：",
    round(patient_region_123_within_one, 3)
)

print(
    "患者・部位単位 QWK：",
    round(patient_region_123_qwk, 3)
)

In [ ]:
# 画像単位と患者・部位単位の評価を比較する
weighted_123_level_comparison = pd.DataFrame(
    {
        "画像単位": [
            weighted_123_accuracy,
            weighted_123_mae,
            weighted_123_within_one,
            weighted_123_qwk
        ],
        "患者・部位単位": [
            patient_region_123_accuracy,
            patient_region_123_mae,
            patient_region_123_within_one,
            patient_region_123_qwk
        ]
    },
    index=[
        "Accuracy",
        "MAE",
        "1点以内正解率",
        "QWK"
    ]
).round(3)

display(weighted_123_level_comparison)

In [ ]:
# 患者・部位単位の混同行列を作る
patient_region_123_confusion = confusion_matrix(
    patient_region_123_true,
    patient_region_123_pred,
    labels=[1, 2, 3]
)

patient_region_123_confusion_df = pd.DataFrame(
    patient_region_123_confusion,
    index=[
        "正解1",
        "正解2",
        "正解3"
    ],
    columns=[
        "予測1",
        "予測2",
        "予測3"
    ]
)

print("【患者・部位単位の混同行列】")
display(patient_region_123_confusion_df)

In [ ]:
# 患者・部位単位のクラス別評価
print("【患者・部位単位のクラス別評価】")

print(
    classification_report(
        patient_region_123_true,
        patient_region_123_pred,
        labels=[1, 2, 3],
        target_names=[
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
### クラス重み付き3クラスMobileNetV2を使用し、同じ患者・同じ部位の複数画像について予測確率を平均して、部位単位の紅斑スコア1～3を決定する。

In [ ]:
# 採用候補の重み付き3クラスモデルで
# テスト用画像を予測する
test_123_probabilities = weighted_model_123.predict(
    test_123_ds,
    verbose=1
)

# 画像単位の予測クラス0・1・2
test_123_predictions = np.argmax(
    test_123_probabilities,
    axis=1
)

# 元の紅斑スコア1・2・3へ戻す
test_123_predicted_scores = (
    test_123_predictions + 1
)

# テストデータの正解スコア
test_123_true_scores = (
    test_123_df["original_score"]
    .astype("int32")
    .to_numpy()
)

print("テスト画像数：", len(test_123_true_scores))
print("予測結果数：", len(test_123_predicted_scores))

assert (
    len(test_123_true_scores)
    == len(test_123_predicted_scores)
)

In [ ]:
# テストデータへ各スコアの予測確率を追加する
test_123_result_df = test_123_df.copy()

test_123_result_df["probability_1"] = (
    test_123_probabilities[:, 0]
)

test_123_result_df["probability_2"] = (
    test_123_probabilities[:, 1]
)

test_123_result_df["probability_3"] = (
    test_123_probabilities[:, 2]
)


# 同じ患者・同じ部位の予測確率を平均する
test_123_patient_region = (
    test_123_result_df
    .groupby(
        ["patient_id", "region"],
        as_index=False
    )
    .agg(
        正解スコア=("original_score", "first"),
        画像数=("image_path", "size"),
        probability_1=("probability_1", "mean"),
        probability_2=("probability_2", "mean"),
        probability_3=("probability_3", "mean")
    )
)


# 平均確率が最大のクラスを選び、
# 元の紅斑スコア1～3へ戻す
probability_columns_123 = [
    "probability_1",
    "probability_2",
    "probability_3"
]

test_123_patient_region["予測スコア"] = (
    np.argmax(
        test_123_patient_region[
            probability_columns_123
        ].to_numpy(),
        axis=1
    )
    + 1
)

In [ ]:
# 患者・部位単位の正解値と予測値
test_patient_region_true = (
    test_123_patient_region["正解スコア"]
    .astype("int32")
    .to_numpy()
)

test_patient_region_pred = (
    test_123_patient_region["予測スコア"]
    .astype("int32")
    .to_numpy()
)


# 最終評価指標
test_accuracy = accuracy_score(
    test_patient_region_true,
    test_patient_region_pred
)

test_mae = mean_absolute_error(
    test_patient_region_true,
    test_patient_region_pred
)

test_within_one = np.mean(
    np.abs(
        test_patient_region_true
        - test_patient_region_pred
    ) <= 1
)

test_qwk = cohen_kappa_score(
    test_patient_region_true,
    test_patient_region_pred,
    weights="quadratic"
)

print(
    "テスト患者・部位数：",
    len(test_patient_region_true)
)

print(
    "テスト Accuracy：",
    round(test_accuracy, 3)
)

print(
    "テスト MAE：",
    round(test_mae, 3)
)

print(
    "テスト 1点以内正解率：",
    round(test_within_one, 3)
)

print(
    "テスト QWK：",
    round(test_qwk, 3)
)

In [ ]:
# テストデータの混同行列
test_confusion = confusion_matrix(
    test_patient_region_true,
    test_patient_region_pred,
    labels=[1, 2, 3]
)

test_confusion_df = pd.DataFrame(
    test_confusion,
    index=["正解1", "正解2", "正解3"],
    columns=["予測1", "予測2", "予測3"]
)

display(test_confusion_df)


# テストデータのクラス別評価
print(
    classification_report(
        test_patient_region_true,
        test_patient_region_pred,
        labels=[1, 2, 3],
        target_names=[
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
# 採用候補のクラス重み付き3クラスモデルで、
# テスト用画像を予測する
test_123_probabilities = weighted_model_123.predict(
    test_123_ds,
    verbose=1
)

print(
    "予測確率の形：",
    test_123_probabilities.shape
)

print(
    "テスト画像数：",
    len(test_123_df)
)

# 予測数とテスト画像数が一致することを確認する
assert (
    len(test_123_probabilities)
    == len(test_123_df)
)

print("テスト画像数と予測数が一致しています。")

In [ ]:
# テストデータへ、紅斑1・2・3の予測確率を追加する
test_123_result_df = test_123_df.copy()

test_123_result_df["probability_1"] = (
    test_123_probabilities[:, 0]
)

test_123_result_df["probability_2"] = (
    test_123_probabilities[:, 1]
)

test_123_result_df["probability_3"] = (
    test_123_probabilities[:, 2]
)


# 同じ患者・同じ部位の複数画像について、
# 各紅斑スコアの予測確率を平均する
test_123_patient_region = (
    test_123_result_df
    .groupby(
        ["patient_id", "region"],
        as_index=False
    )
    .agg(
        正解スコア=("original_score", "first"),
        画像数=("image_path", "size"),
        probability_1=("probability_1", "mean"),
        probability_2=("probability_2", "mean"),
        probability_3=("probability_3", "mean")
    )
)

In [ ]:
# 紅斑1・2・3の平均予測確率を配列にする
test_probability_columns = [
    "probability_1",
    "probability_2",
    "probability_3"
]

# 最も平均確率が高いクラスを選び、
# 元の紅斑スコア1・2・3へ変換する
test_123_patient_region["予測スコア"] = (
    np.argmax(
        test_123_patient_region[
            test_probability_columns
        ].to_numpy(),
        axis=1
    )
    + 1
)

# 正解との絶対誤差を計算する
test_123_patient_region["絶対誤差"] = np.abs(
    test_123_patient_region["正解スコア"]
    - test_123_patient_region["予測スコア"]
)

# 完全に一致したかを記録する
test_123_patient_region["正解"] = (
    test_123_patient_region["正解スコア"]
    == test_123_patient_region["予測スコア"]
)

display(
    test_123_patient_region.head(20)
)

In [ ]:
# 患者・部位単位の正解スコアを取り出す
test_patient_region_true = (
    test_123_patient_region["正解スコア"]
    .astype("int32")
    .to_numpy()
)

# 患者・部位単位の予測スコアを取り出す
test_patient_region_pred = (
    test_123_patient_region["予測スコア"]
    .astype("int32")
    .to_numpy()
)


# 完全一致率
test_accuracy = accuracy_score(
    test_patient_region_true,
    test_patient_region_pred
)

# 正解と予測の平均点差
test_mae = mean_absolute_error(
    test_patient_region_true,
    test_patient_region_pred
)

# 正解との差が1点以内だった割合
test_within_one = np.mean(
    np.abs(
        test_patient_region_true
        - test_patient_region_pred
    ) <= 1
)

# スコアの順序と誤差の大きさを考慮した一致度
test_qwk = cohen_kappa_score(
    test_patient_region_true,
    test_patient_region_pred,
    weights="quadratic"
)


print(
    "テスト患者・部位数：",
    len(test_patient_region_true)
)

print(
    "テスト Accuracy：",
    round(test_accuracy, 3)
)

print(
    "テスト MAE：",
    round(test_mae, 3)
)

print(
    "テスト 1点以内正解率：",
    round(test_within_one, 3)
)

print(
    "テスト QWK：",
    round(test_qwk, 3)
)

In [ ]:
# テストデータの正解数と予測数を比較する
test_distribution = pd.DataFrame({
    "正解数": (
        pd.Series(test_patient_region_true)
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    ),
    "予測数": (
        pd.Series(test_patient_region_pred)
        .value_counts()
        .reindex([1, 2, 3], fill_value=0)
    )
})

test_distribution.index.name = "紅斑スコア"

print("【テストデータの正解数と予測数】")
display(test_distribution)

In [ ]:
# テストデータの患者・部位単位の混同行列
test_confusion = confusion_matrix(
    test_patient_region_true,
    test_patient_region_pred,
    labels=[1, 2, 3]
)

test_confusion_df = pd.DataFrame(
    test_confusion,
    index=[
        "正解1",
        "正解2",
        "正解3"
    ],
    columns=[
        "予測1",
        "予測2",
        "予測3"
    ]
)

print("【テストデータの混同行列】")
display(test_confusion_df)

In [ ]:
# テストデータのクラス別評価
print("【テストデータのクラス別評価】")

print(
    classification_report(
        test_patient_region_true,
        test_patient_region_pred,
        labels=[1, 2, 3],
        target_names=[
            "紅斑1",
            "紅斑2",
            "紅斑3"
        ],
        digits=3,
        zero_division=0
    )
)

In [ ]:
# 最終テスト結果を表にまとめる
final_test_metrics = pd.DataFrame(
    {
        "評価値": [
            test_accuracy,
            test_mae,
            test_within_one,
            test_qwk,
            0.434
        ]
    },
    index=[
        "Accuracy",
        "MAE",
        "1点以内正解率",
        "QWK",
        "Macro F1"
    ]
).round(3)

display(final_test_metrics)

In [ ]:
from sklearn.metrics import f1_score

test_macro_f1 = f1_score(
    test_patient_region_true,
    test_patient_region_pred,
    labels=[1, 2, 3],
    average="macro",
    zero_division=0
)

final_test_metrics = pd.DataFrame(
    {
        "評価値": [
            test_accuracy,
            test_mae,
            test_within_one,
            test_qwk,
            test_macro_f1
        ]
    },
    index=[
        "Accuracy",
        "MAE",
        "1点以内正解率",
        "QWK",
        "Macro F1"
    ]
).round(3)

display(final_test_metrics)

In [ ]:
# 完成モデルの重みファイルを確認する
print(
    "重みファイルの存在：",
    weighted_123_path.is_file()
)

print(
    "完成モデルの重み保存場所：",
    weighted_123_path
)

In [ ]:
# 患者・部位ごとの最終テスト結果を保存する
test_result_path = (
    model_dir
    / "erythema_123_test_patient_region_results.csv"
)

test_123_patient_region.to_csv(
    test_result_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "患者・部位別テスト結果の保存場所：",
    test_result_path
)

In [ ]:
# 最終評価指標を保存する
metrics_path = (
    model_dir
    / "erythema_123_test_metrics.csv"
)

final_test_metrics.to_csv(
    metrics_path,
    encoding="utf-8-sig"
)

print(
    "最終評価指標の保存場所：",
    metrics_path
)

In [ ]:
# 患者ごとのデータ分割を保存する
split_records = []

for patient_id in sorted(train_123_patients):
    split_records.append({
        "patient_id": patient_id,
        "split": "train"
    })

for patient_id in sorted(val_123_patients):
    split_records.append({
        "patient_id": patient_id,
        "split": "validation"
    })

for patient_id in sorted(test_123_patients):
    split_records.append({
        "patient_id": patient_id,
        "split": "test"
    })

split_123_df = pd.DataFrame(split_records)

split_path = (
    model_dir
    / "erythema_123_patient_split.csv"
)

split_123_df.to_csv(
    split_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "患者分割の保存場所：",
    split_path
)

紅斑スコア0は患者・部位単位で6件のみであり、上肢および下肢には該当データが存在しなかったため、紅斑陽性部位を対象としたスコア1～3の3クラス分類モデルを構築した。患者単位で学習・検証・テストデータを分割し、学習用データでは同一患者・同一部位から使用する画像を最大10枚に制限した。ImageNetで事前学習されたMobileNetV2を特徴抽出器として用い、平方根で緩和したクラス重みを適用した。同一患者・同一部位の複数画像について予測確率を平均し、部位単位の予測スコアを算出した。独立テストデータ78患者・部位におけるAccuracyは0.628、MAEは0.372、1点以内正解率は1.000、Quadratic Weighted Kappaは0.250、Macro F1は0.434であった。紅斑1、2、3のRecallは、それぞれ0.133、0.846、0.273であり、紅斑2に比べて紅斑1および3の判別性能が低いことが課題として残った。

In [ ]:
# テストデータの患者・部位単位で、
# 紅斑1・2・3のOne-vs-Rest ROC曲線を描き、
# 高解像度PNGとして保存する

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc


# 患者・部位単位の正解スコア
y_true_roc = (
    test_123_patient_region["正解スコア"]
    .astype("int32")
    .to_numpy()
)


# 患者・部位単位の予測確率
# 列の順番は紅斑1、紅斑2、紅斑3
y_score_roc = (
    test_123_patient_region[
        [
            "probability_1",
            "probability_2",
            "probability_3"
        ]
    ]
    .to_numpy()
)


# 正解スコア1・2・3を、
# One-vs-Rest用の0・1形式へ変換する
y_true_binary = label_binarize(
    y_true_roc,
    classes=[1, 2, 3]
)


# ROC曲線上で使用するクラス名
class_names_roc = {
    0: "Erythema 1",
    1: "Erythema 2",
    2: "Erythema 3"
}


# クラスごとのROC曲線とAUCを保存する辞書
fpr = {}
tpr = {}
roc_auc = {}


# 紅斑1・2・3について、
# 各クラス対その他のROC曲線を計算する
for class_index in range(3):

    fpr[class_index], tpr[class_index], _ = roc_curve(
        y_true_binary[:, class_index],
        y_score_roc[:, class_index]
    )

    roc_auc[class_index] = auc(
        fpr[class_index],
        tpr[class_index]
    )


# Micro-average ROC
# 全患者・部位、全クラスの判定をまとめて評価する
fpr["micro"], tpr["micro"], _ = roc_curve(
    y_true_binary.ravel(),
    y_score_roc.ravel()
)

roc_auc["micro"] = auc(
    fpr["micro"],
    tpr["micro"]
)


# Macro-average ROC
# 紅斑1・2・3を同じ重みで平均する
all_fpr = np.unique(
    np.concatenate(
        [
            fpr[class_index]
            for class_index in range(3)
        ]
    )
)

mean_tpr = np.zeros_like(all_fpr)

for class_index in range(3):

    mean_tpr += np.interp(
        all_fpr,
        fpr[class_index],
        tpr[class_index]
    )

mean_tpr /= 3

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr

roc_auc["macro"] = auc(
    fpr["macro"],
    tpr["macro"]
)


# ROC曲線を描く
plt.figure(
    figsize=(9, 8)
)


# クラスごとの線の色
colors = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:green"
}


# 紅斑1・2・3のROC曲線を描く
for class_index in range(3):

    plt.plot(
        fpr[class_index],
        tpr[class_index],
        color=colors[class_index],
        linewidth=2,
        label=(
            f"{class_names_roc[class_index]} "
            f"(AUC = {roc_auc[class_index]:.3f})"
        )
    )


# Macro-average ROCを描く
plt.plot(
    fpr["macro"],
    tpr["macro"],
    color="tab:red",
    linestyle="--",
    linewidth=2.5,
    label=(
        f"Macro-average "
        f"(AUC = {roc_auc['macro']:.3f})"
    )
)


# Micro-average ROCを描く
plt.plot(
    fpr["micro"],
    tpr["micro"],
    color="tab:purple",
    linestyle=":",
    linewidth=2.5,
    label=(
        f"Micro-average "
        f"(AUC = {roc_auc['micro']:.3f})"
    )
)


# ランダム予測の基準線を描く
plt.plot(
    [0, 1],
    [0, 1],
    color="gray",
    linestyle="--",
    linewidth=1.5,
    label="Chance level (AUC = 0.500)"
)


# 軸の表示範囲を指定する
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])


# 軸ラベルとグラフタイトル
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curves for Erythema Severity Classification\n"
    "Patient-Region Level, One-vs-Rest"
)


# 凡例と補助線を表示する
plt.legend(
    loc="lower right"
)

plt.grid(
    alpha=0.25
)


# タイトルや凡例が重ならないよう配置を調整する
plt.tight_layout()


# PNGファイルの保存場所を指定する
roc_png_path = (
    model_dir
    / "erythema_123_test_roc_curve.png"
)


# Notebookで表示する前に、
# ROC曲線を高解像度PNGとして保存する
plt.savefig(
    roc_png_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


# Notebook上にもROC曲線を表示する
plt.show()


# 保存結果を確認する
print(
    "ROC曲線の保存場所：",
    roc_png_path
)

print(
    "PNGファイルの存在：",
    roc_png_path.is_file()
)


# クラス別・平均AUCを表にまとめる
roc_auc_table = pd.DataFrame(
    {
        "AUC": [
            roc_auc[0],
            roc_auc[1],
            roc_auc[2],
            roc_auc["macro"],
            roc_auc["micro"]
        ]
    },
    index=[
        "紅斑1 vs その他",
        "紅斑2 vs その他",
        "紅斑3 vs その他",
        "Macro-average",
        "Micro-average"
    ]
).round(3)


# AUC一覧を表示する
display(roc_auc_table)

In [ ]:
# クラス別・平均AUCを表にまとめる
roc_auc_table = pd.DataFrame(
    {
        "AUC": [
            roc_auc[0],
            roc_auc[1],
            roc_auc[2],
            roc_auc["macro"],
            roc_auc["micro"]
        ]
    },
    index=[
        "紅斑1 vs その他",
        "紅斑2 vs その他",
        "紅斑3 vs その他",
        "Macro-average",
        "Micro-average"
    ]
).round(3)

display(roc_auc_table)

患者・部位単位の予測確率を用いてOne-vs-Rest ROC解析を行った。紅斑スコア1、2、3のAUCは、それぞれ0.637、0.558、0.768であり、Macro-average AUCは0.665、Micro-average AUCは0.798であった。紅斑3では比較的高い識別能を示した一方、紅斑2のAUCは0.558であり、他クラスとの確率的な分離能は限定的であった。クラス数に偏りがあるため、全サンプルをまとめたMicro-averageより、クラスを等しく扱うMacro-averageおよびクラス別AUCを重視した。

### Grad Cam

In [ ]:
# Grad-CAMに必要な変数が残っていることを確認する
required_names = [
    "weighted_model_123",
    "test_123_df",
    "test_123_probabilities",
    "model_dir",
    "IMG_SIZE"
]

for name in required_names:
    print(
        name,
        "：",
        name in globals()
    )

In [ ]:
# weighted_model_123のレイヤー構成を表示する
for index, layer in enumerate(
    weighted_model_123.layers
):
    print(
        index,
        layer.name,
        type(layer).__name__,
        getattr(layer, "output_shape", "")
    )

In [ ]:
# weighted_model_123から必要なレイヤーを取得する

gradcam_augmentation = next(
    (
        layer
        for layer in weighted_model_123.layers
        if isinstance(layer, tf.keras.Sequential)
    ),
    None
)

gradcam_base_model = next(
    (
        layer
        for layer in weighted_model_123.layers
        if isinstance(layer, tf.keras.Model)
        and "mobilenetv2" in layer.name.lower()
    ),
    None
)

gradcam_pooling = next(
    (
        layer
        for layer in weighted_model_123.layers
        if isinstance(
            layer,
            tf.keras.layers.GlobalAveragePooling2D
        )
    ),
    None
)

gradcam_dropout = next(
    (
        layer
        for layer in weighted_model_123.layers
        if isinstance(
            layer,
            tf.keras.layers.Dropout
        )
    ),
    None
)

gradcam_classifier = next(
    (
        layer
        for layer in reversed(
            weighted_model_123.layers
        )
        if isinstance(
            layer,
            tf.keras.layers.Dense
        )
    ),
    None
)


# 必要なレイヤーを取得できたか確認する
gradcam_layers = {
    "データ拡張": gradcam_augmentation,
    "MobileNetV2": gradcam_base_model,
    "Pooling": gradcam_pooling,
    "Dropout": gradcam_dropout,
    "分類層": gradcam_classifier
}

for label, layer in gradcam_layers.items():
    print(
        label,
        "：",
        None if layer is None else layer.name
    )

if any(
    layer is None
    for layer in gradcam_layers.values()
):
    raise ValueError(
        "Grad-CAMに必要なレイヤーを"
        "すべて取得できませんでした。"
    )

In [ ]:
# 画像1枚からGrad-CAMヒートマップを作成する

def make_gradcam_heatmap(
    image_batch,
    class_index=None
):
    """
    class_index:
        0 = 紅斑1
        1 = 紅斑2
        2 = 紅斑3
    """

    with tf.GradientTape() as tape:

        # 学習時専用のランダム変形を無効にする
        x = gradcam_augmentation(
            image_batch,
            training=False
        )

        # MobileNetV2用の画像前処理
        x = tf.keras.applications.mobilenet_v2.preprocess_input(
            x
        )

        # MobileNetV2の最終特徴マップ
        feature_maps = gradcam_base_model(
            x,
            training=False
        )

        tape.watch(feature_maps)

        # 完成モデルと同じ分類処理
        x = gradcam_pooling(
            feature_maps
        )

        x = gradcam_dropout(
            x,
            training=False
        )

        predictions = gradcam_classifier(
            x
        )

        # 指定がなければ、予測確率が最大のクラスを対象にする
        if class_index is None:
            class_index = int(
                tf.argmax(
                    predictions[0]
                ).numpy()
            )

        target_score = predictions[
            :,
            class_index
        ]

    # 対象クラスに対する特徴マップの勾配
    gradients = tape.gradient(
        target_score,
        feature_maps
    )

    if gradients is None:
        raise ValueError(
            "Grad-CAM用の勾配を計算できませんでした。"
        )

    # 特徴チャンネルごとの重要度
    pooled_gradients = tf.reduce_mean(
        gradients,
        axis=(0, 1, 2)
    )

    feature_maps = feature_maps[0]

    # チャンネルごとの特徴と重要度を掛け合わせる
    heatmap = tf.reduce_sum(
        feature_maps
        * pooled_gradients,
        axis=-1
    )

    # 正の寄与だけを残す
    heatmap = tf.maximum(
        heatmap,
        0
    )

    # 0～1に正規化する
    heatmap_max = tf.reduce_max(
        heatmap
    )

    if float(heatmap_max.numpy()) > 0:
        heatmap = (
            heatmap / heatmap_max
        )

    return (
        heatmap.numpy(),
        predictions.numpy()[0],
        class_index
    )

In [ ]:
# Grad-CAM表示用に画像を読み込む

def load_gradcam_image(image_path):

    image = tf.keras.utils.load_img(
        image_path,
        target_size=(
            IMG_SIZE,
            IMG_SIZE
        )
    )

    image_array = (
        tf.keras.utils.img_to_array(
            image
        )
    )

    display_image = (
        image_array
        .clip(0, 255)
        .astype("uint8")
    )

    image_batch = np.expand_dims(
        image_array,
        axis=0
    ).astype("float32")

    return display_image, image_batch

In [ ]:
# ヒートマップを元画像へ重ねる

def overlay_gradcam(
    display_image,
    heatmap,
    alpha=0.40
):

    resized_heatmap = tf.image.resize(
        heatmap[..., np.newaxis],
        (
            display_image.shape[0],
            display_image.shape[1]
        )
    ).numpy().squeeze()

    colored_heatmap = plt.get_cmap(
        "jet"
    )(resized_heatmap)[
        ...,
        :3
    ]

    colored_heatmap = (
        colored_heatmap * 255
    ).astype("uint8")

    overlay = (
        (1 - alpha)
        * display_image.astype("float32")
        + alpha
        * colored_heatmap.astype("float32")
    )

    return (
        overlay
        .clip(0, 255)
        .astype("uint8")
    )

In [ ]:
# テスト画像ごとの正解・予測結果を作る
gradcam_test_df = test_123_df.copy()

gradcam_test_df["probability_1"] = (
    test_123_probabilities[:, 0]
)

gradcam_test_df["probability_2"] = (
    test_123_probabilities[:, 1]
)

gradcam_test_df["probability_3"] = (
    test_123_probabilities[:, 2]
)

gradcam_test_df["predicted_score"] = (
    np.argmax(
        test_123_probabilities,
        axis=1
    )
    + 1
)

gradcam_test_df["confidence"] = np.max(
    test_123_probabilities,
    axis=1
)

gradcam_test_df["correct"] = (
    gradcam_test_df["original_score"]
    == gradcam_test_df["predicted_score"]
)

print(
    "正解画像数：",
    gradcam_test_df["correct"].sum()
)

print(
    "誤分類画像数：",
    (~gradcam_test_df["correct"]).sum()
)

In [ ]:
# 予測確率が高い正解画像
correct_example = (
    gradcam_test_df[
        gradcam_test_df["correct"]
    ]
    .sort_values(
        "confidence",
        ascending=False
    )
    .iloc[0]
)

# 予測確率が高い誤分類画像
incorrect_example = (
    gradcam_test_df[
        ~gradcam_test_df["correct"]
    ]
    .sort_values(
        "confidence",
        ascending=False
    )
    .iloc[0]
)

print("正解例：")
display(
    correct_example[
        [
            "patient_id",
            "region",
            "original_score",
            "predicted_score",
            "confidence"
        ]
    ]
)

print("誤分類例：")
display(
    incorrect_example[
        [
            "patient_id",
            "region",
            "original_score",
            "predicted_score",
            "confidence"
        ]
    ]
)

In [ ]:
# 正解例の画像を読み込む
display_image, image_batch = load_gradcam_image(
    correct_example["image_path"]
)

# 予測されたクラスをGrad-CAM対象にする
predicted_class_index = (
    int(correct_example["predicted_score"])
    - 1
)

# Grad-CAMヒートマップを作る
heatmap, probabilities, used_class_index = (
    make_gradcam_heatmap(
        image_batch=image_batch,
        class_index=predicted_class_index
    )
)

# 元画像へ重ねる
overlay_image = overlay_gradcam(
    display_image,
    heatmap,
    alpha=0.40
)

# 3枚を表示する
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

axes[0].imshow(display_image)
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(
    heatmap,
    cmap="jet",
    vmin=0,
    vmax=1
)
axes[1].set_title("Grad-CAM heatmap")
axes[1].axis("off")

axes[2].imshow(overlay_image)
axes[2].set_title("Grad-CAM overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print(
    "正解スコア：",
    int(correct_example["original_score"])
)

print(
    "予測スコア：",
    int(correct_example["predicted_score"])
)

print(
    "予測確率：",
    round(
        float(
            probabilities[
                used_class_index
            ]
        ),
        3
    )
)

In [ ]:
# テストデータの患者・部位単位で、
# 紅斑1・2・3ごとのTP・FP・FN・TNを計算する

import numpy as np
import pandas as pd

from sklearn.metrics import multilabel_confusion_matrix


# 正解スコアと予測スコア
y_true_final = test_patient_region_true
y_pred_final = test_patient_region_pred

# 紅斑1・2・3について、
# One-vs-Rest形式の混同行列を作る
class_confusion_matrices = multilabel_confusion_matrix(
    y_true_final,
    y_pred_final,
    labels=[1, 2, 3]
)


# クラスごとの結果を保存する
confusion_records = []

for score, matrix in zip(
    [1, 2, 3],
    class_confusion_matrices
):
    # 各クラスの混同行列は次の配置
    #
    # [[TN, FP],
    #  [FN, TP]]

    tn, fp, fn, tp = matrix.ravel()

    confusion_records.append({
        "紅斑スコア": score,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn)
    })


# 表にする
tp_fp_fn_tn_table = (
    pd.DataFrame(confusion_records)
    .set_index("紅斑スコア")
)

display(tp_fp_fn_tn_table)

In [ ]:
# 0除算を防ぎながら割合を計算する関数
def safe_divide(numerator, denominator):
    if denominator == 0:
        return np.nan

    return numerator / denominator


# クラスごとの詳細指標を計算する
metric_records = []

for score, matrix in zip(
    [1, 2, 3],
    class_confusion_matrices
):
    tn, fp, fn, tp = matrix.ravel()

    # 感度・Recall・True Positive Rate
    sensitivity = safe_divide(
        tp,
        tp + fn
    )

    # 特異度・True Negative Rate
    specificity = safe_divide(
        tn,
        tn + fp
    )

    # 適合率・Positive Predictive Value
    precision = safe_divide(
        tp,
        tp + fp
    )

    # 陰性的中率・Negative Predictive Value
    npv = safe_divide(
        tn,
        tn + fn
    )

    # F1-score
    f1 = safe_divide(
        2 * precision * sensitivity,
        precision + sensitivity
    )

    # One-vs-Rest Accuracy
    class_accuracy = safe_divide(
        tp + tn,
        tp + tn + fp + fn
    )

    # False Positive Rate
    fpr_value = safe_divide(
        fp,
        fp + tn
    )

    # False Negative Rate
    fnr_value = safe_divide(
        fn,
        fn + tp
    )

    metric_records.append({
        "紅斑スコア": score,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "感度_Recall": sensitivity,
        "特異度": specificity,
        "適合率_Precision": precision,
        "陰性的中率_NPV": npv,
        "F1": f1,
        "One-vs-Rest Accuracy": class_accuracy,
        "偽陽性率_FPR": fpr_value,
        "偽陰性率_FNR": fnr_value
    })


# 表へ変換する
class_metrics_table = (
    pd.DataFrame(metric_records)
    .set_index("紅斑スコア")
)

# 小数点以下3桁へ丸める
rate_columns = [
    "感度_Recall",
    "特異度",
    "適合率_Precision",
    "陰性的中率_NPV",
    "F1",
    "One-vs-Rest Accuracy",
    "偽陽性率_FPR",
    "偽陰性率_FNR"
]

class_metrics_table[
    rate_columns
] = class_metrics_table[
    rate_columns
].round(3)

display(class_metrics_table)

In [ ]:
# クラス別指標を同じ重みで平均する
macro_metrics = (
    class_metrics_table[
        rate_columns
    ]
    .mean()
    .to_frame(
        name="Macro-average"
    )
    .round(3)
)

display(macro_metrics)

In [ ]:
# クラス別TP・FP・FN・TNと評価指標をCSV保存する
class_metrics_path = (
    model_dir
    / "erythema_123_test_class_metrics.csv"
)

class_metrics_table.to_csv(
    class_metrics_path,
    encoding="utf-8-sig"
)

print(
    "クラス別評価指標の保存場所：",
    class_metrics_path
)

print(
    "CSVファイルの存在：",
    class_metrics_path.is_file()
)

感度 = TP / (TP + FN)

特異度 = TN / (TN + FP)

適合率 = TP / (TP + FP)

陰性的中率 = TN / (TN + FN)

F1 = 2 × 適合率 × 感度 / (適合率 + 感度)

偽陽性率 = FP / (FP + TN)

偽陰性率 = FN / (FN + TP)

### GradCAMの画像

In [ ]:
# 指定した紅斑スコアについて、
# テスト画像をTP・FP・FN・TNへ分類する

def classify_gradcam_examples(
    dataframe,
    target_score
):
    """
    target_scoreについてOne-vs-Restで分類する

    TP：正解も予測もtarget_score
    FP：正解は別だが予測がtarget_score
    FN：正解はtarget_scoreだが予測は別
    TN：正解も予測もtarget_score以外
    """

    result_df = dataframe.copy()

    # 正解が対象スコアか
    true_is_target = (
        result_df["original_score"]
        == target_score
    )

    # 予測が対象スコアか
    predicted_is_target = (
        result_df["predicted_score"]
        == target_score
    )

    # TP・FP・FN・TNの条件
    conditions = [
        true_is_target & predicted_is_target,
        ~true_is_target & predicted_is_target,
        true_is_target & ~predicted_is_target,
        ~true_is_target & ~predicted_is_target
    ]

    categories = [
        "TP",
        "FP",
        "FN",
        "TN"
    ]

    result_df["prediction_type"] = np.select(
        conditions,
        categories,
        default="Unclassified"
    )

    return result_df

In [ ]:
# TP・FP・FN・TNごとにGrad-CAM画像を表示し、
# 1枚のPNGとして保存する

def show_gradcam_examples(
    dataframe,
    target_score,
    prediction_type,
    max_images=4
):
    """
    target_score:
        1、2、3

    prediction_type:
        "TP"、"FP"、"FN"、"TN"

    max_images:
        表示する最大画像数
    """

    # TP・FP・FN・TNへ分類する
    classified_df = classify_gradcam_examples(
        dataframe=dataframe,
        target_score=target_score
    )

    # 指定区分だけを取り出す
    all_examples = classified_df[
        classified_df["prediction_type"]
        == prediction_type
    ].copy()

    print(
        f"紅斑{target_score}・{prediction_type}：",
        len(all_examples),
        "画像"
    )

    if all_examples.empty:
        print("該当する画像はありません。")
        return None

    # TP・FPの場合は対象クラスの予測確率が高い順
    if prediction_type in ["TP", "FP"]:

        probability_column = (
            f"probability_{target_score}"
        )

        examples = all_examples.sort_values(
            probability_column,
            ascending=False
        )

    # FNの場合は、対象クラスの確率が低い順
    elif prediction_type == "FN":

        probability_column = (
            f"probability_{target_score}"
        )

        examples = all_examples.sort_values(
            probability_column,
            ascending=True
        )

    # TNの場合はモデルの予測確率が高い順
    else:
        examples = all_examples.sort_values(
            "confidence",
            ascending=False
        )

    # 最大表示枚数までにする
    examples = examples.head(
        max_images
    ).reset_index(drop=True)

    number_of_images = len(examples)

    # 1画像につき、元画像とGrad-CAMの2列を作る
    fig, axes = plt.subplots(
        number_of_images,
        2,
        figsize=(
            10,
            4.5 * number_of_images
        )
    )

    # 1枚の場合も2次元配列として扱う
    if number_of_images == 1:
        axes = np.expand_dims(
            axes,
            axis=0
        )

    for row_number, example in examples.iterrows():

        # 画像を読み込む
        display_image, image_batch = (
            load_gradcam_image(
                example["image_path"]
            )
        )

        # Grad-CAMでは、実際にモデルが予測したクラスを対象にする
        predicted_score = int(
            example["predicted_score"]
        )

        predicted_class_index = (
            predicted_score - 1
        )

        # ヒートマップを作る
        heatmap, probabilities, used_class_index = (
            make_gradcam_heatmap(
                image_batch=image_batch,
                class_index=predicted_class_index
            )
        )

        # 元画像へGrad-CAMを重ねる
        overlay_image = overlay_gradcam(
            display_image,
            heatmap,
            alpha=0.40
        )

        selected_probability = float(
            probabilities[
                used_class_index
            ]
        )

        # 元画像
        axes[row_number, 0].imshow(
            display_image
        )

        axes[row_number, 0].set_title(
            (
                "Original image\n"
                f"Patient: {example['patient_id']} | "
                f"Region: {example['region']}"
            ),
            fontsize=10
        )

        axes[row_number, 0].axis("off")

        # Grad-CAM重ね合わせ画像
        axes[row_number, 1].imshow(
            overlay_image
        )

        axes[row_number, 1].set_title(
            (
                f"Grad-CAM: {prediction_type}\n"
                f"True: {int(example['original_score'])} | "
                f"Predicted: {predicted_score} | "
                f"Probability: {selected_probability:.3f}"
            ),
            fontsize=10
        )

        axes[row_number, 1].axis("off")

    fig.suptitle(
        (
            f"Erythema {target_score}: "
            f"{prediction_type} Grad-CAM examples"
        ),
        fontsize=15
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.96]
    )

    # PNG保存先
    output_path = (
        model_dir
        / (
            f"erythema_{target_score}_"
            f"{prediction_type.lower()}_gradcam.png"
        )
    )

    # 表示前に保存する
    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    print(
        "保存場所：",
        output_path
    )

    print(
        "PNGファイルの存在：",
        output_path.is_file()
    )

    return examples

In [ ]:
# 紅斑3のTrue Positive画像へGrad-CAMを適用する
erythema_3_tp_examples = show_gradcam_examples(
    dataframe=gradcam_test_df,
    target_score=3,
    prediction_type="TP",
    max_images=4
)

In [ ]:
# 紅斑3のFalse Positive
erythema_3_fp_examples = show_gradcam_examples(
    dataframe=gradcam_test_df,
    target_score=3,
    prediction_type="FP",
    max_images=4
)

In [ ]:
# 紅斑3のFalse Negative
erythema_3_fn_examples = show_gradcam_examples(
    dataframe=gradcam_test_df,
    target_score=3,
    prediction_type="FN",
    max_images=4
)

In [ ]:
# 紅斑3のTrue Negative
erythema_3_tn_examples = show_gradcam_examples(
    dataframe=gradcam_test_df,
    target_score=3,
    prediction_type="TN",
    max_images=4
)

In [ ]:
# 紅斑1・2・3について、
# TP・FP・FN・TNのGrad-CAM画像を一括作成する

for target_score in [1, 2, 3]:

    for prediction_type in [
        "TP",
        "FP",
        "FN",
        "TN"
    ]:

        show_gradcam_examples(
            dataframe=gradcam_test_df,
            target_score=target_score,
            prediction_type=prediction_type,
            max_images=4
        )

In [ ]:
import sys

print(sys.version)

In [ ]:
import platform

print(platform.python_version())

In [ ]:
# 指定スコアについて、画像をTP・FP・FN・TNに分類する
def classify_gradcam_cases(dataframe, target_score):

    result_df = dataframe.copy()

    true_is_target = (
        result_df["original_score"] == target_score
    )

    predicted_is_target = (
        result_df["predicted_score"] == target_score
    )

    conditions = [
        true_is_target & predicted_is_target,    # TP
        ~true_is_target & predicted_is_target,   # FP
        true_is_target & ~predicted_is_target,   # FN
        ~true_is_target & ~predicted_is_target   # TN
    ]

    choices = ["TP", "FP", "FN", "TN"]

    result_df["prediction_type"] = np.select(
        conditions,
        choices,
        default="未分類"
    )

    return result_df


In [ ]:
# Grad-CAM表示用の候補画像を選ぶ
def select_gradcam_candidates(
    dataframe,
    target_score,
    prediction_type,
    max_images=8,
    selection="high_confidence"
):

    classified_df = classify_gradcam_cases(
        dataframe=dataframe,
        target_score=target_score
    )

    candidates = classified_df[
        classified_df["prediction_type"] == prediction_type
    ].copy()

    if candidates.empty:
        print(
            f"紅斑{target_score}の"
            f"{prediction_type}画像はありません。"
        )
        return candidates

    target_probability_column = (
        f"probability_{target_score}"
    )

    if selection == "high_confidence":

        if prediction_type in ["TP", "FP"]:
            # 対象スコアを強く予測した順
            candidates = candidates.sort_values(
                target_probability_column,
                ascending=False
            )

        elif prediction_type == "FN":
            # 本当は対象スコアなのに、その確率が低かった順
            candidates = candidates.sort_values(
                target_probability_column,
                ascending=True
            )

        else:
            # TNは実際の予測クラスへの確信度が高い順
            candidates = candidates.sort_values(
                "confidence",
                ascending=False
            )

    elif selection == "random":
        candidates = candidates.sample(
            frac=1,
            random_state=42
        )

    else:
        raise ValueError(
            "selectionは'high_confidence'または"
            "'random'を指定してください。"
        )

    # 同じ患者・部位から何枚も選ばない
    candidates = candidates.drop_duplicates(
        subset=["patient_id", "region"]
    )

    candidates = (
        candidates
        .head(max_images)
        .reset_index(drop=True)
    )

    print(
        f"紅斑{target_score}・{prediction_type}："
        f"全{len(classified_df[classified_df['prediction_type'] == prediction_type])}枚"
    )

    print(
        f"表示候補：{len(candidates)}枚"
    )

    return candidates


In [ ]:
# 複数候補の元画像とGrad-CAMを一覧表示する
def display_gradcam_candidates(
    candidates,
    target_score,
    prediction_type,
    save_name=None
):

    if candidates.empty:
        print("表示できる候補がありません。")
        return

    number_of_images = len(candidates)

    fig, axes = plt.subplots(
        number_of_images,
        2,
        figsize=(10, 4.2 * number_of_images)
    )

    # 候補が1枚でも2次元配列として扱う
    if number_of_images == 1:
        axes = np.expand_dims(
            axes,
            axis=0
        )

    for row_index, candidate in candidates.iterrows():

        display_image, image_batch = load_gradcam_image(
            candidate["image_path"]
        )

        predicted_score = int(
            candidate["predicted_score"]
        )

        predicted_class_index = predicted_score - 1

        heatmap, probabilities, used_class_index = (
            make_gradcam_heatmap(
                image_batch=image_batch,
                class_index=predicted_class_index
            )
        )

        overlay_image = overlay_gradcam(
            display_image,
            heatmap,
            alpha=0.40
        )

        predicted_probability = float(
            probabilities[used_class_index]
        )

        # 元画像
        axes[row_index, 0].imshow(
            display_image
        )

        axes[row_index, 0].set_title(
            (
                f"Original | Candidate {row_index + 1}\n"
                f"Patient: {candidate['patient_id']} | "
                f"Region: {candidate['region']}"
            ),
            fontsize=10
        )

        axes[row_index, 0].axis("off")

        # Grad-CAM重ね合わせ
        axes[row_index, 1].imshow(
            overlay_image
        )

        axes[row_index, 1].set_title(
            (
                f"{prediction_type} | "
                f"True: {int(candidate['original_score'])} | "
                f"Pred: {predicted_score}\n"
                f"Probability: {predicted_probability:.3f}"
            ),
            fontsize=10
        )

        axes[row_index, 1].axis("off")

    fig.suptitle(
        (
            f"Erythema {target_score}: "
            f"{prediction_type} Grad-CAM candidates"
        ),
        fontsize=15
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.98]
    )

    if save_name is not None:

        output_path = (
            model_dir
            / save_name
        )

        plt.savefig(
            output_path,
            dpi=200,
            bbox_inches="tight",
            facecolor="white"
        )

        print(
            "候補一覧の保存場所：",
            output_path
        )

    plt.show()

In [ ]:
erythema3_tp_candidates = select_gradcam_candidates(
    dataframe=gradcam_test_df,
    target_score=3,
    prediction_type="TP",
    max_images=8,
    selection="high_confidence"
)

display_gradcam_candidates(
    candidates=erythema3_tp_candidates,
    target_score=3,
    prediction_type="TP",
    save_name="gradcam_candidates_erythema3_tp.png"
)

In [ ]:
erythema3_fn_candidates = select_gradcam_candidates(
    dataframe=gradcam_test_df,
    target_score=3,
    prediction_type="FN",
    max_images=8,
    selection="high_confidence"
)

display_gradcam_candidates(
    candidates=erythema3_fn_candidates,
    target_score=3,
    prediction_type="FN",
    save_name="gradcam_candidates_erythema3_fn.png"
)

In [ ]:
erythema2_tp_candidates = select_gradcam_candidates(
    dataframe=gradcam_test_df,
    target_score=2,
    prediction_type="TP",
    max_images=20,
    selection="high_confidence"
)

display_gradcam_candidates(
    candidates=erythema2_tp_candidates,
    target_score=2,
    prediction_type="TP",
    save_name="gradcam_candidates_erythema2_tp.png"
)

In [ ]:
erythema3_fn_candidates = select_gradcam_candidates(
    dataframe=gradcam_test_df,
    target_score=3,
    prediction_type="FN",
    max_images=8,
    selection="high_confidence"
)

display_gradcam_candidates(
    candidates=erythema3_fn_candidates,
    target_score=3,
    prediction_type="FN",
    save_name="gradcam_candidates_erythema3_fn.png"
)